# Environment

In [ ]:
# =========================
# Standard library
# =========================
import os
import re
from pathlib import Path
from typing import List, Tuple, Dict

# =========================
# Third-party libraries
# =========================
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm.auto import tqdm

# =========================
# Project-specific modules
# =========================
from kbase.config import settings
import kbase.preprocessing as dp
import kbase.modeling as ml 

# Show all rows when displaying value_counts() / dataframes
pd.set_option("display.max_rows", None)



# Main Vectors

In [ ]:
key_col = "demandpk"
id_columns = ["healthcard_id", "national_id", "full_name"]

# Data Loading

## EDATOSCASO-LLAMADAS

In [ ]:
df_triaje = pq.read_table(os.path.join(settings.source_tables_path, settings.master_table_4demands_path)).to_pandas()
df_triaje.shape

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(10)


# Run Pipeline in subet

In [ ]:
# Flag to test the pipeline in a subset (change subset in kbase/preprocessing.py)
## must be either "all", to run in all dataframe, or "test", to run it in a subset
df_triaje = dp.df_pipeline_test(df_triaje).sort_values(by="DEMANDAPK")
n_rows_source_data = len(df_triaje)

# Column Reduction


In [ ]:
# IDTIPODEMANDA1 and IDTIPODEMANDA2: additional information outside of the scope

# Columns to drop
columns_to_drop = []

# Drop columns
if columns_to_drop:
    df_triaje = df_triaje.drop(columns=columns_to_drop)
    print(f"Dropped {len(columns_to_drop):,} columns.")
else:
    print("No columns were dropped.")
# df_triaje.info()


# Data filtering by demand type

In [ ]:
# Rename columns and summarize numbers before filtering by demand type
df_triaje = dp.rename_columns(df_triaje)
dp.summarize_numbers(df_triaje, key_col, id_columns)


In [ ]:
df_triaje.head(5)

In [ ]:
df_triaje['demand_type_1'].value_counts(dropna = False)

In [ ]:
# Exclusion of rows that do not correspond to the target demand type (stroke, code 54)
n_rows_before_demand_filter = len(df_triaje)
print(f"Number of rows before filtering by demand type: {n_rows_before_demand_filter:,}")

df_triaje = df_triaje[df_triaje['demand_type_1']==54]

n_rows_after_demand_filter = len(df_triaje)
print(f"Number of rows after filtering by demand type: {n_rows_after_demand_filter:,}")

n_rows_excluded_demand_filter = n_rows_before_demand_filter - n_rows_after_demand_filter
print(
    "Rows removed after filtering by demand type:",
    n_rows_excluded_demand_filter
)

In [ ]:
# Summarize numbers after filtering by demand type
dp.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
df_triaje.sample(5)

# Data cleaning

## DEMANDAPK - demandpk

In [ ]:
# Step 1: Count how many unique patients are linked to each demandpk
demand_patient_counts = (
    df_triaje.groupby('demandpk')['patientid']
    .nunique()
    .reset_index(name='n_patients')
)

# Step 2: Count distribution (optional)
distribution = (
    demand_patient_counts['n_patients']
    .value_counts(dropna=False)
    .sort_index()
)
print(distribution)

# ===============================================================
# Step 3: Identify demandpk with >= 2 linked patients
# ===============================================================
demandpk_mult = demand_patient_counts.loc[
    demand_patient_counts['n_patients'] >= 2, 
    'demandpk'
]

print(f"\nDemandPK with >=2 patients: {len(demandpk_mult):,}")

# ===============================================================
# Step 4: Remove rows belonging to those demandpk
# ===============================================================
n_rows_before_demandpk_exclusion = len(df_triaje)

df_triaje = df_triaje[~df_triaje['demandpk'].isin(demandpk_mult)].reset_index(drop=True)

n_rows_after_demandpk_exclusion = len(df_triaje)

# Number of rows eliminated
n_rows_excluded_demandpk = n_rows_before_demandpk_exclusion - n_rows_after_demandpk_exclusion

print(f"Rows excluded due to duplicate demandpk (>=2 patients): {n_rows_excluded_demandpk:,}")
print(f"Rows after excluding duplicate demandpk: {n_rows_after_demandpk_exclusion:,}")

dp.summarize_numbers(df_triaje, key_col, id_columns)


## FECHADEMANDA - demand_date

In [ ]:
# Convert the raw string date
df_triaje["demand_date"] = pd.to_datetime(
    df_triaje["demand_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "demand_date" rows: {df_triaje["demand_date"].isna().sum():,}')

df_triaje["demand_date"].dtype



## FNACIMIENTO - birth_date

In [ ]:
# Convert the raw string date
df_triaje["birth_date"] = pd.to_datetime(
    df_triaje["birth_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "birth_date" rows: {df_triaje["birth_date"].isna().sum():,}')

df_triaje["birth_date"].dtype

## EDAD - age

In [ ]:
df_triaje["age_type"].value_counts(dropna = False)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index(ascending=True)

In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()



In [ ]:

def process_age_pipeline(df, birth_col="birth_date", demand_col="demand_date",
                         age_col="age", age_type_col="age_type"):
    """
    Full age processing pipeline applied directly on `age` column.

    Steps:
    1. Compute age from birth_date when available.
    2. Compute age from reported age + age_type (A/M/D/S) when birth_date is missing.
    3. Replace the original 'age' column with the best available estimate.
    4. Round age to nearest integer.
    5. Convert final age to Int8.
    6. Remove auxiliary processing columns.

    Returns:
        The dataframe with an updated 'age' column (Int8).
    """

    # ------------------------------------------------------------
    # Step 1 — Compute age in years using birth_date
    # ------------------------------------------------------------
    df["age_from_birthdate"] = (
        (df[demand_col] - df[birth_col])
        .dt.total_seconds() / (365.25 * 24 * 3600)
    )

    # ------------------------------------------------------------
    # Step 2 — Define conversion multipliers for age_type
    # ------------------------------------------------------------
    unit_to_years = {
        "A": 1.0,        # years
        "M": 1/12,       # months
        "D": 1/365.25,   # days
        "S": 1/52,       # weeks
    }

    # ------------------------------------------------------------
    # Step 3 — Normalize age_type into uppercase strings
    # ------------------------------------------------------------
    df[age_type_col] = df[age_type_col].astype("string").str.upper()

    # ------------------------------------------------------------
    # Step 4 — Compute reported age in years using age and age_type
    # ------------------------------------------------------------
    df["reported_age_years"] = df[age_col] * df[age_type_col].map(unit_to_years)

    # ------------------------------------------------------------
    # Step 5 — Mask for rows missing birth_date but having reported age
    # ------------------------------------------------------------
    mask_missing_birth = (
        df[birth_col].isna() &
        df[age_col].notna() &
        (df[age_col] != 0)
    )

    # ------------------------------------------------------------
    # Step 6 — Update age directly
    # ------------------------------------------------------------
    # Start with age_from_birthdate when available
    df[age_col] = df["age_from_birthdate"]

    # Replace missing birthdate ages with reported_age_years
    df.loc[mask_missing_birth, age_col] = df.loc[mask_missing_birth, "reported_age_years"]

    # ------------------------------------------------------------
    # Step 7 — Round final age to nearest integer
    # ------------------------------------------------------------
    df[age_col] = df[age_col].round()

    # ------------------------------------------------------------
    # Step 8 — Convert to Int16
    # ------------------------------------------------------------
    df[age_col] = df[age_col].astype("Int16")

    # ------------------------------------------------------------
    # Step 9 — Remove auxiliary columns
    # ------------------------------------------------------------
    df = df.drop(columns=["age_from_birthdate", "reported_age_years"])

    return df

df_triaje = process_age_pipeline(df_triaje)


In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()

In [ ]:
## Exclusion of outliers and filtering by age range of interest
n_rows_before_age_exclusion = len(df_triaje)
print(f"Number of rows before age filtering: {n_rows_before_age_exclusion:,}")

df_triaje = df_triaje[df_triaje["age"].between(15, 105)]

n_rows_after_age_exclusion = len(df_triaje)

n_rows_excluded_age = n_rows_before_age_exclusion - n_rows_after_age_exclusion
print(f"Number of rows after age filtering: {n_rows_after_age_exclusion:,}")

print("Rows removed after excluding outliers and filtering by age range:",
      n_rows_excluded_age)


In [ ]:
# df_triaje_bebe = df_triaje[df_triaje["age"]==0]

In [ ]:
# df_triaje_bebe.head(100)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index().reset_index()


In [ ]:
df_triaje['age'].hist(bins=20, range=(0, 100), color='skyblue', edgecolor='black')
plt.title('Histogram of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6) 
plt.show()

df_triaje.boxplot(column=['age'], by='demand_type_1')
plt.title('Boxplot of Age')
plt.ylabel('Age')
plt.show()


In [ ]:
# WHO age-group indicators (binary 0/1, mutually exclusive and collectively exhaustive)
age_groups = {
    "age_15_24":   (15, 24),
    "age_25_44":   (25, 44),
    "age_45_59":   (45, 59),
    "age_60_74":   (60, 74),
    "age_75_plus": (75, 105),
}

for col, (lo, hi) in age_groups.items():
    df_triaje[col] = df_triaje["age"].between(lo, hi).astype("Int8")

# QC: each row must belong to exactly one age group
age_group_cols = list(age_groups.keys())
group_sums = df_triaje[age_group_cols].sum(axis=1)
n_invalid = (group_sums != 1).sum()
print(f"QC - rows not assigned to exactly one age group: {n_invalid:,}")
assert n_invalid == 0, "Age group indicators are not mutually exclusive and collectively exhaustive"

df_triaje[age_group_cols].sum().rename("count")

## SEX - sex

In [ ]:
print("Value counts of 'sex' column before conversion:")
print(df_triaje['sex'].value_counts(dropna=False))

In [ ]:
# Map dictionary
## 'D' and None values are unknown
## Male (H): 0
## Female (M): 1
map_sex = {
    'D': np.nan,
    'None': np.nan,
    'H': 0,
    'M': 1
}

# Map the values
df_triaje["sex"] = df_triaje["sex"].map(map_sex)

# Convert data type
df_triaje["sex"] = pd.to_numeric(df_triaje["sex"], errors="coerce")
df_triaje["sex"] = df_triaje["sex"].astype("Int64")
df_triaje["sex"] = df_triaje["sex"].astype("category")

# Check the mapping
print("\nValue counts of 'sex' column after data conversion:")
df_triaje["sex"].value_counts(dropna=False)


In [ ]:
## Exclusion of None
n_rows_before_sex_exclusion = len(df_triaje)
print(f"Number of rows before filtering out missing values in sex: {n_rows_before_sex_exclusion:,}")

# Filter out rows where sex is NaN
df_triaje = df_triaje[df_triaje["sex"].notna()]

n_rows_after_sex_exclusion = len(df_triaje)
print(f"Number of rows after filtering out missing values in sex: {n_rows_after_sex_exclusion:,}")

n_rows_excluded_sex = n_rows_before_sex_exclusion - n_rows_after_sex_exclusion

print(
    "Rows removed after filtering out missing values in sex:",
    n_rows_excluded_sex
)


In [ ]:
print("\nValue counts of 'sex' column after removing missing values:")
df_triaje["sex"].value_counts(dropna=False)


## IDASISTENCIA

In [ ]:
df_triaje["assistance_id"].value_counts(dropna=False)

## Time columns

In [ ]:
# List of time columns
time_columns = [
    "contact_entry_time",
    "contact_answer_time",
    "activation_time",
    "departure_time",
    "arrival_time",
    "load_time",
    "operative_time",
    "destination_time",
    "available_time"
]

for col in time_columns:
    df_triaje[col] = pd.to_datetime(
        df_triaje[col],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

# Count NaT values after conversion
print("\nCount of invalid or missing datetime values after conversion:")
for col in time_columns:
    nat_count = df_triaje[col].isna().sum()
    dtype = df_triaje[col].dtype
    print(f"{col} ({dtype}): {nat_count:,} NaT values")


## IDPROVINCIA - province

In [ ]:
# Original value counts
print("Value counts of 'province' column before conversion:")
df_triaje["province"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_province = {
    4 : "ALM",
    11: "CAD",
    14: "COR",
    18: "GRA",
    21: "HUE",
    23: "JAE",
    29: "MAL",
    41: "SEV"
}

# Map the values
df_triaje["province"] = df_triaje["province"].map(map_province)

# Check the mapping
print("Value counts of 'province' column after conversion:")
df_triaje["province"].value_counts(dropna=False)

# Map to categorical numerical 
map_province_num = {
    "ALM": 0,
    "CAD": 1,
    "COR": 2,
    "GRA": 3,
    "HUE": 4,
    "JAE": 5,
    "MAL": 6,
    "SEV": 7
}

# Map the values
df_triaje["province"] = df_triaje["province"].map(map_province_num)

# Check the mapping
print("Value counts of 'province' column after conversion:")
df_triaje["province"].value_counts(dropna=False)


## Location coordinates

In [ ]:
# ---------------------------------------------------------
# Exclusion of Missing Coordinates (Latitude & Longitude)
# ---------------------------------------------------------
# n_rows_before_coords = len(df_triaje)
# print(f"Number of rows before coordinate exclusion: {n_rows_before_coords}")

# # Filter out rows where latitude or longitude are NaN
# df_triaje = df_triaje.dropna(subset=["incident_latitude", "incident_longitude"])

# n_rows_after_coords = len(df_triaje)
# n_row_excluded_coords = n_rows_before_coords - n_rows_after_coords

# print(f"Number of rows after coordinate exclusion: {n_rows_after_coords}")
# print(f"Rows removed after filtering out missing coordinates: {n_row_excluded_coords}")

## resource_dest_province_id

In [ ]:
# Original value counts
print("Value counts of 'resource_dest_province_id' column before conversion:")
df_triaje["resource_dest_province_id"].value_counts(dropna=False)


## IDALERTANTE - alert_receiver

This variable provides us with an identification code for the entity that received the alert. We have a total of 23 different IDs (as we will verify below). According to clinical indications, they will be regrouped into the following 6 groups:

- GROUP 1 '112': '112'.
- GROUP 2 'USER': 'USER' (User).
- GROUP 3 'POL/FG' (Police/Fire Brigade): '91', '92', '62', '80', '091', '092', '062', '080'.
- GROUP 4 'HS' (Health System): 'APS', 'SEU', 'SNU', 'HOS', '61', '061', 'ECA', 'PAC', 'DCC'.
- GROUP 5 'TELE' (Teleassistance): 'SAT'.
- GROUP 6 'OTHERS': 'PCI', 'OTR', 'SR', 'CRE', 'FAX', 'TEL', 'INA', 'RES'.

In [ ]:
# Original value counts
print("Value counts of 'alert_receiver' column before conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

In [ ]:
# Map raw alert receiver codes to grouped categories.
def map_alert_receiver(x):
    """
    This function:
    - Normalizes input values (strip spaces, uppercase).
    - Handles None/NaN safely.
    - Uses a dictionary-based mapping for maintainability.
    - Returns np.nan when the code is unknown or invalid.
    """

    # If null → return NaN
    if pd.isna(x):
        return np.nan

    # Normalize input (string only)
    x = str(x).strip().upper()
    if x == "":
        return np.nan

    # Dictionary of mappings
    mapping_dict = {
        # Group 1
        "112": "112",

        # Group 2
        "USU": "USER",

        # Group 3: POL/FG
        "91": "POL/FG", "92": "POL/FG", "62": "POL/FG", "80": "POL/FG",
        "091": "POL/FG", "092": "POL/FG", "062": "POL/FG", "080": "POL/FG",

        # Group 4: HS
        "APS": "HS", "SEU": "HS", "SNU": "HS", "HOS": "HS",
        "61": "HS", "061": "HS", "ECA": "HS", "PAC": "HS", "DCC": "HS",

        # Group 5: TELE
        "SAT": "TELE",

        # Group 6: OTHERS
        "PCI": "OTHERS", "OTR": "OTHERS", "SR": "OTHERS", "CRE": "OTHERS",
        "FAX": "OTHERS", "TEL": "OTHERS", "INA": "OTHERS", "RES": "OTHERS",
    }

    # Return mapped value or NaN if not found
    return mapping_dict.get(x, np.nan)


# Map the values
df_triaje["alert_receiver"] = df_triaje["alert_receiver"].apply(map_alert_receiver)

# Check the mapping
print("Value counts of 'alert_receiver' column after conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

In [ ]:
# Mapping to categorical numerical values
map_receiver_numeric = {
    "112": 0,
    "USER": 1,
    "POL/FG": 2,
    "HS": 3,
    "TELE": 4,
    "OTHERS": 5
}

# Map the values
df_triaje["alert_receiver"] = df_triaje["alert_receiver"].map(map_receiver_numeric)

# 4. Final verification
print("Value counts of encoded 'alert_receiver':")
print(df_triaje["alert_receiver"].value_counts(dropna=False).sort_index())

In [ ]:
# Store counts before exclusion
n_rows_before_alert_exclusion = len(df_triaje)
print(f"Number of rows before filtering out missing values in alert_receiver: {n_rows_before_alert_exclusion:,}")

# 2. Filter out rows where alert_receiver_encoded is NaN
df_triaje = df_triaje[df_triaje["alert_receiver"].notna()]

# 3. Store counts after exclusion
n_rows_after_alert_exclusion = len(df_triaje)
print(f"Number of rows after filtering out missing values in alert_receiver: {n_rows_after_alert_exclusion:,}")

# 4. Calculate total excluded
n_rows_excluded_alert = n_rows_before_alert_exclusion - n_rows_after_alert_exclusion

print(
    "Rows removed after filtering out missing values in alert_receiver:",
    n_rows_excluded_alert
)

## IDESCENARIO - location_patient
- GROUP 1 'HO' (Home): 'DO'
- GROUP 2 'PR' (Public Road): 'VP'
- GROUP 3 'OTHERS': 'PI', 'CA', 'OT'.

In [ ]:
# Original value counts
print("Value counts of 'location_patient' column before conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

In [ ]:
# Show distribution before mapping
print("Value counts of 'location_patient' before mapping:")
print(df_triaje["location_patient"].value_counts(dropna=False))

# Map original codes to readable categories
map_location = {
    "DO": "ho",
    "VP": "pr",
    "CA": "other",
    "OT": "other",
    "PI": "other"
}
df_triaje["location_patient"] = df_triaje["location_patient"].map(map_location)
print(df_triaje["location_patient"].value_counts(dropna=False))

# Encode categories to numeric values (0, 1, 2)
map_numeric = {
    "ho": 0,
    "pr": 1,
    "other": 2
}
df_triaje["location_patient"] = df_triaje["location_patient"].map(map_numeric)

# Final check
print("\nValue counts after numeric encoding:")
print(df_triaje["location_patient"].value_counts(dropna=False))

## IDTIPORECURSO - resource_type

In [ ]:
# Original value counts
print("Value counts of 'resource_type' column before conversion:")
df_triaje["resource_type"].value_counts(dropna=False)



In [ ]:
print(f"\nNumber of unique values in resource_type: {df_triaje['resource_type'].nunique():,}")

## IDTARJETACORAZON - healthcard

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False).head(20)

In [ ]:
print(f"Number of None values in 'healthcard' is: {df_triaje['healthcard'].isna().sum():,}")
df_triaje["healthcard"] = df_triaje["healthcard"].notna().astype(int)
df_triaje["healthcard"].dtype

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False)

## IDTARJETAAIRE - aircard

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

In [ ]:
print(f"Number of None values in 'aircard' is: {df_triaje['aircard'].isna().sum():,}")
df_triaje["aircard"] = df_triaje["aircard"].notna().astype(int)
df_triaje["aircard"].dtype

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

## ESVULNERABLE - is_vulnerable

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

In [ ]:
df_triaje["is_vulnerable"] = df_triaje["is_vulnerable"].notna().astype(int)
df_triaje["is_vulnerable"].dtype

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

## IDHCDM

In [ ]:
df_triaje["hcdm_id"].isna().sum()

In [ ]:
df_triaje["hcdm_id"].notna().sum()

In [ ]:
print(df_triaje["hcdm_id"].nunique())

In [ ]:
# This filters for non-nulls first, THEN picks 10 random actual values
df_triaje[df_triaje["hcdm_id"].notna()]["hcdm_id"].sample(10)

In [ ]:
# Definimos el patrón a buscar
search_pattern = "f08b4e60bfb0"

# Filtramos y mostramos las primeras filas que coinciden
df_matches_triaje = df_triaje[df_triaje['hcdm_id'].str.contains(search_pattern, na=False, case=False)]

print(f"Coincidencias encontradas: {len(df_matches_triaje):,}")
display(df_matches_triaje.head())

## MOTIVOLITERAL - literal_reason

In [ ]:

def clean_medical_text(text):
    """
    Final version: Removes real and literal carriage returns (\\r\\n), 
    processes Teleassistance metadata, and handles empty values.
    """
    if pd.isna(text) or str(text).strip().lower() in ["", "none", "nan"]:
        return "sin descripcion"
    
    # Ensure it's string and lowercase
    text = str(text).lower()
    
    # 1. REMOVE LITERAL AND REAL NEWLINES
    # First, replace literal strings like '\r', '\n' or '\r\n' (text representations)
    text = text.replace('\\r', ' ').replace('\\n', ' ')
    # Second, replace actual control characters (non-visible breaks)
    text = re.sub(r'[\r\n]+', ' ', text)

    # 2. Process Teleassistance patterns
    if re.match(r'^\d{2}/\d{2}/\d{4}', text):
        # Pattern A: Standard teleassistance with "afectado" and ";"
        tele_full_pattern = r'^\d{2}/\d{2}/\d{4}.*?afectado:?[^;]+;'
        # Pattern B: Direct date-operator format (e.g., ICTUS example)
        tele_short_pattern = r'^\d{2}/\d{2}/\d{4}.*?[:\.]\s*'
        
        if "afectado" in text and ";" in text:
            text = re.sub(tele_full_pattern, 'teleasistencia - ', text)
        else:
            text = re.sub(tele_short_pattern, 'teleasistencia - ', text)

    # 3. Remove hardware labels and 9+ digit numbers (Phones/IDs)
    text = text.replace('nº tel dispositivo', '')
    text = re.sub(r'\d{9,}', '', text)

    # 4. Final cleaning of multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 5. Final validation
    if text in ["", "none", "nan"]:
        return "sin descripcion"
    
    return text

df_triaje = df_triaje.rename(columns = {"literal_reason": "literal_reason_raw"})

# --- Apply and Verify ---
df_triaje['literal_reason'] = df_triaje['literal_reason_raw'].apply(clean_medical_text)

# Check specifically for those literal strings
check_remnants = df_triaje[df_triaje['literal_reason'].str.contains(r'\\r|\\n', na=False)]
print(f"Remaining literal breaks: {len(check_remnants):,}")

In [ ]:
# QC on literal_reason column

# Filter and create a copy for the new table
# ^\d means: "starts with a digit"
df_numeric_start = df_triaje[df_triaje['literal_reason_raw'].str.contains(r'^\d', na=False)].copy()

# Verify the result
print(f"Rows found: {len(df_numeric_start):,}")

# Display a sample of the raw vs cleaned text along with the primary key
display(df_numeric_start[['demandpk', 'literal_reason_raw', 'literal_reason']].sample(100))

# Drop the raw text column to save memory
df_triaje = df_triaje.drop(columns=['literal_reason_raw'])

## Tests

In [ ]:
df_triaje.info()

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.sample(50)

# Joining tables

## triage questions

In [ ]:
# Load table
df_preguntas = pq.read_table(os.path.join(settings.source_tables_path, 
                                          "preguntas_triaje/preguntas_respuesta_detalle_20260122.parquet")).to_pandas()
df_preguntas.shape

In [ ]:
df_preguntas.info()

In [ ]:
df_preguntas.head(100)

In [ ]:
# Test value counts
df_preguntas["Q7"].value_counts(dropna = False)

In [ ]:
# Test value counts
df_preguntas["Q5"].value_counts(dropna = False)

In [ ]:
# Fill missing values with "0"
questions_cols = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]
df_preguntas.loc[:, questions_cols] = df_preguntas.loc[:, questions_cols].fillna("0")



In [ ]:
# Initial count of total rows and unique IDs
print(f"Rows in df_triaje before joining df_preguntas: {len(df_triaje):,}")
print(f"Rows in df_preguntas is: {len(df_preguntas):,}")
print(f"\nNumber of unique DEMANDAPK in df_preguntas before deduplication: {df_preguntas['DEMANDAPK'].nunique():,}")

# Calculate the number of duplicate rows in df_preguntas based on DEMANDAPK
n_duplicated = df_preguntas["DEMANDAPK"].duplicated().sum()
print(f"Number of duplicated DEMANDAPK in df_preguntas is: {n_duplicated:,}")

# Filter df_preguntas to include only IDs present in df_triaje to avoid fan-out
# and remove duplicates from DEMANDAPK to prevent row multiplication during merge
df_preguntas_unique = df_preguntas[df_preguntas["DEMANDAPK"].isin(df_triaje["demandpk"])]
df_preguntas_unique = df_preguntas_unique.drop_duplicates(subset=["DEMANDAPK"])

# Count unique IDs after filtering and deduplication process
print(f"Number of unique DEMANDAPK in df_preguntas after deduplication: {df_preguntas_unique['DEMANDAPK'].nunique():,}")

# Left join df_triaje with the deduplicated questions dataframe
df_triaje = df_triaje.merge(
    df_preguntas_unique[["DEMANDAPK"] + questions_cols],
    how="left",
    left_on="demandpk",
    right_on="DEMANDAPK"
)

# Drop the redundant join key
df_triaje = df_triaje.drop(columns=["DEMANDAPK"])

# Fill missing values in question columns with "0"
df_triaje.loc[:, questions_cols] = df_triaje.loc[:, questions_cols].fillna("0")

# Rename the question columns to lowercase
df_triaje = df_triaje.rename(
    columns={col: col.lower() for col in questions_cols}
)

# Summarize overall numbers
dp.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
df_triaje.head(5)

In [ ]:
# Test value counts
df_triaje["q5"].value_counts(dropna = False)

In [ ]:
dp.summarize_numbers(df_triaje, key_col, id_columns)

df_triaje.info()

## ICD from hospital - BPS origin

In [ ]:
# Loading exported table
df_hosp = pd.read_excel(os.path.join(settings.source_tables_path, "datos_BPS_Jun2026/Datos_HOSPITALIZACIONES_CRUZADOS.xlsx"))
df_hosp.shape

In [ ]:
df_hosp.info()

In [ ]:
df_hosp.head(100)

In [ ]:
df_hosp["DemandaPk"].nunique()

In [ ]:
df_urg = pd.read_excel(os.path.join(settings.source_tables_path, "datos_BPS_Jun2026/Datos_URGENCIAS_CRUZADOS.xlsx"))

In [ ]:
df_urg.info()

In [ ]:
df_urg.head(100)

In [ ]:
df_urg["DemandaPk"].nunique()

In [ ]:
# Build a wide table with one row per demandpk and all hospitalization ICD codes
# (COD_CIE), ordered by their original position (COD_POSICION), deduplicated and
# joined by '|'
df_hosp_icd = (
    df_hosp.rename(columns={"DemandaPk": "demandpk"})
    .sort_values(["demandpk", "COD_POSICION"])
    .drop_duplicates(subset=["demandpk", "COD_CIE"])
    .groupby("demandpk")["COD_CIE"]
    .apply("|".join)
    .reset_index()
    .rename(columns={"COD_CIE": "icd_codes_hosp"})
)
df_hosp_icd.shape

In [ ]:
df_hosp_icd.head(10)

In [ ]:
# Build a wide table with one row per demandpk and all emergency-visit ICD codes
# (COD_CIE_DIAGNOSTICO), deduplicated and joined by '|'
df_urg_icd = (
    df_urg.rename(columns={"DemandaPk": "demandpk"})
    .drop_duplicates(subset=["demandpk", "COD_CIE_DIAGNOSTICO"])
    .groupby("demandpk")["COD_CIE_DIAGNOSTICO"]
    .apply("|".join)
    .reset_index()
    .rename(columns={"COD_CIE_DIAGNOSTICO": "icd_codes_urg"})
)
df_urg_icd.shape

In [ ]:
df_urg_icd.head(10)

In [ ]:
# Outer merge on demandpk: rows present in only one table keep that table's codes,
# rows present in both combine the codes from both sources without duplicates
df_icd_bps = df_hosp_icd.merge(df_urg_icd, on="demandpk", how="outer")

def combine_icd_codes(row):
    codes_hosp = row["icd_codes_hosp"].split("|") if pd.notna(row["icd_codes_hosp"]) else []
    codes_urg = row["icd_codes_urg"].split("|") if pd.notna(row["icd_codes_urg"]) else []
    return "|".join(dict.fromkeys(codes_hosp + codes_urg))

df_icd_bps["icd_codes_bps"] = df_icd_bps.apply(combine_icd_codes, axis=1)
df_icd_bps = df_icd_bps[["demandpk", "icd_codes_bps"]]

assert df_icd_bps["demandpk"].is_unique
df_icd_bps.shape

In [ ]:
df_icd_bps.head(10)

In [ ]:
# Join ICD codes from hospital/emergency BPS sources into df_triaje by demandpk
df_triaje = df_triaje.merge(df_icd_bps, on="demandpk", how="left")
df_triaje.shape

In [ ]:
# Flag rows with at least one ICD code from the BPS data source
df_triaje["has_icd_bps"] = df_triaje["icd_codes_bps"].notna().astype("int8")

In [ ]:
df_triaje["has_icd_bps"].value_counts(dropna=False)

## Literal reason - Motivo literal

In [ ]:
# Load table
df_lr_stroke = pq.read_table(os.path.join(
    settings.source_tables_path, 
    "motivo_literal_ictus/df_stroke_motivo_literal_onehot_anon_2026-08-19.parquet")).to_pandas()
df_lr_stroke.shape


In [ ]:
df_lr_stroke.head()
print(df_lr_stroke.info())

In [ ]:
# Handling NA values
print(f"Number of unique hcdm_id: {df_lr_stroke['hcdm_id'].nunique():,}")
print(f"Number of NaN in hcdm_id column: {df_lr_stroke['hcdm_id'].isna().sum():,}")

# Remove rows where 'hcdm_id' is NaN and overwrite the dataframe
df_lr_stroke = df_lr_stroke.dropna(subset=['hcdm_id'])

# Verification
print(f"Number of unique hcdm_id after removing NA values: {df_lr_stroke['hcdm_id'].nunique():,}")
print(f"Number of NaN in hcdm_id column removing NA values: {df_lr_stroke['hcdm_id'].isna().sum():,}")


In [ ]:
# QC: check for duplicate hcdm_id in df_lr_stroke
duplicate_count_lr = df_lr_stroke["hcdm_id"].duplicated().sum()

if duplicate_count_lr > 0:
    print(f"⚠️ WARNING: Found {duplicate_count_lr:,} duplicate hcdm_id in df_lr_stroke.")
    duplicates_lr = df_lr_stroke[df_lr_stroke["hcdm_id"].duplicated(keep=False)]
    with pd.option_context("display.max_rows", 10, "display.max_columns", None):
        display(duplicates_lr.sort_values(by="hcdm_id").head(100))
else:
    print("✅ No duplicate hcdm_id found in df_lr_stroke.")

In [ ]:
# Group by hcdm_id: max for ml_* columns; rename ml_* -> lr_*
ml_cols = [col for col in df_lr_stroke.columns if col.startswith("ml_")]
# Convert lr_ columns to int8
df_lr_stroke[ml_cols] = df_lr_stroke[ml_cols].astype("int8") 

df_lr_stroke_grouped = (
    df_lr_stroke
    .groupby("hcdm_id", as_index=False)[ml_cols]
    .max()
    .rename(columns={col: col.replace("ml_", "lr_", 1) for col in ml_cols})
)

print(f"Rows before groupby: {len(df_lr_stroke):,}")
print(f"Rows after groupby:  {len(df_lr_stroke_grouped):,}")
df_lr_stroke_grouped.head()

In [ ]:
def lr_mergeand_audit(df_left, df_right, on_col, lr_prefix="lr_", fill_val=0):
    """
    Performs a left join and prints detailed matching statistics.
    Capsules all temporary variables to keep the global environment clean.
    """
    # 1. Identify columns to merge (the key + columns starting with prefix)
    target_cols = [col for col in df_right.columns if col.startswith(lr_prefix)]
    
    # 2. Perform the merge
    df_result = df_left.merge(
        df_right[[on_col] + target_cols],
        how="left",
        on=on_col,
        indicator=True
    )
   
    # 3. Calculate statistics (internal to the function)
    stats = df_result["_merge"].value_counts()
    matched = stats.get("both", 0)
    unmatched = stats.get("left_only", 0)
    total = len(df_result)
    rate = (matched / total) * 100 if total > 0 else 0
    
    # 4. Print report
    print(f"--- Join Statistics (Key: {on_col}) ---")
    print(f"✅ Success matches : {matched:,} rows")
    print(f"❌ No match found  : {unmatched:,} rows")
    print(f"📊 Match rate      : {rate:.2f}%")
    
    # 5. Cleanup: Fill NaNs and remove indicator
    df_result[target_cols] = df_result[target_cols].fillna(fill_val)
    df_result = df_result.drop(columns=["_merge"])
    
    print(f"Final shape: {df_result.shape}\n")
    
    return df_result

# --- HOW TO USE ---
df_triaje = lr_mergeand_audit(df_triaje, df_lr_stroke_grouped, on_col="hcdm_id")

In [ ]:
def translate_lr_columns(df, translation_dict=dp.lr_translation_dict):
    """
    Renames lr_* columns from Spanish abbreviations to English using translation_dict.

    Warns about columns present in df but missing from the dict (left untouched),
    and dict keys absent from df (possible drift). Asserts column count is unchanged.
    Returns the renamed dataframe.
    """
    lr_cols_before = [c for c in df.columns if c.startswith("lr_")]

    not_in_dict = [c for c in lr_cols_before if c not in translation_dict]
    not_in_df   = [k for k in translation_dict if k not in df.columns]

    df = df.rename(columns=translation_dict)

    lr_cols_after = [c for c in df.columns if c.startswith("lr_")]
    n_translated  = len(lr_cols_before) - len(not_in_dict)

    print("--- lr_* column translation ---")
    print(f"✅ Translated       : {n_translated:>3,} column(s)")
    if not_in_dict:
        print(f"⚠️  No dict entry    : {len(not_in_dict):>3,} column(s) — left untouched: {not_in_dict}")
    else:
        print(f"✅ No dict entry    : {len(not_in_dict):>3,} column(s)")
    if not_in_df:
        print(f"⚠️  Dict key missing : {len(not_in_df):>3,} key(s) not in dataframe: {not_in_df}")
    else:
        print(f"✅ Dict key missing : {len(not_in_df):>3,} key(s)")

    if len(lr_cols_before) == len(lr_cols_after):
        print(f"✅ Column count     : {len(lr_cols_before):,} before = {len(lr_cols_after):,} after")
    else:
        raise AssertionError(
            f"❌ Column count changed after translation: {len(lr_cols_before)} -> {len(lr_cols_after)}"
        )

    return df

# Call the function
df_triaje = translate_lr_columns(df_triaje)

## Past history - Antecedentes (new current hist_ columns)


In [ ]:
# Load table
df_history_new = pq.read_table(os.path.join(settings.source_tables_path, 
                                          "antecedentes_stroke/antecedentes_one_hot_encoding_stroke_2026-08-17.parquet")).to_pandas()
df_history_new.shape


In [ ]:
df_history_new.head()

In [ ]:
df_history_new.info()

In [ ]:
df_history_new["tiene_texto"].value_counts(dropna = False)

In [ ]:
df_history_new["demandpk"].nunique()

In [ ]:
# Left join df_triaje with df_history_new

ant_cols_new = [col for col in df_history_new.columns if col.startswith('ant_')]

# Rename ant_* columns to hist_*
# Added in final iteration: translate newly added chronic care plan column.
hist_translation_dict = {
    "ant_9_1_plan_cronicos": "hist_9_1_chronic_care_plan",
}

df_history_new = df_history_new.rename(
    columns={col: hist_translation_dict.get(col, col.replace('ant_', 'hist_', 1)) for col in ant_cols_new}
)

def _hist_sort_key(col):
    parts = col.split('_')
    group = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 999
    item = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 999
    return (group, item, col)

hist_cols_new = sorted(
    [col for col in df_history_new.columns if col.startswith('hist_')],
    key=_hist_sort_key,
)

df_triaje = df_triaje.merge(
    df_history_new[['demandpk', 'tiene_texto'] + hist_cols_new],
    how='left',
    on ='demandpk')

# Creation of has_history variable
df_triaje = df_triaje.rename(columns = {"tiene_texto": "has_history"})

# Fill 0 for unmatched rows and convert hist_* and has_history to int8
cols_history_new = hist_cols_new + ["has_history"]
df_triaje[cols_history_new] = (
    df_triaje[cols_history_new]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype("int8")
)

print(f"Rows with history:    {df_triaje['has_history'].sum():,}")
print(f"Rows without history: {(df_triaje['has_history'] == 0).sum():,}")
df_triaje.head()

## Past history - Antecedentes (old legacy source for com_ columns)


In [ ]:
# Load old legacy history table
df_history_old = pq.read_table(os.path.join(settings.source_tables_path, 
                                          "medicacion_asistencia_antecedentes/df_stroke_antecedentes_one_hot_encoding_2026-06-26.parquet")).to_pandas()
df_history_old.shape


In [ ]:
df_history_old.head()


In [ ]:
df_history_old.info()


In [ ]:
# 1. Extract the last part of the UUID in the old History (Master) table
df_history_old['MATCH_KEY'] = df_history_old['ID_ASISTENCIA_FK'].apply(lambda x: x.split('-')[-1])

# --- SECURITY VERIFICATION (CRITICAL) ---
duplicate_count_history_old = df_history_old['MATCH_KEY'].duplicated().sum()

if duplicate_count_history_old > 0:
    print(f"⚠️ WARNING: Found {duplicate_count_history_old:,} duplicate keys in the old Master table.")
    
    # IDENTIFY AND SORT DUPLICATES
    # 'keep=False' ensures we see ALL occurrences of the repeated keys
    duplicates_history_old_df = df_history_old[df_history_old['MATCH_KEY'].duplicated(keep=False)]
    
    # Sort by MATCH_KEY so duplicates appear one after another
    duplicates_history_old_sorted = duplicates_history_old_df.sort_values(by='MATCH_KEY')
    
    print(f"\n--- Displaying sorted duplicates ---")
    
    # Display the first 100 rows (consecutive) to audit the duplicates
    with pd.option_context('display.max_rows', 10, 'display.max_columns', None):
        display(duplicates_history_old_sorted.head(100))
else:
    print("✅ No duplicate keys found in the old Master table.")

print("Old master table preparation completed.")


In [ ]:
# 2. Groupby MATCH_KEY: max for ant_* columns, date-only for FECHA, comma-join for CATEGORY

# Strip time from FECHA (keep date only)
df_history_old['FECHA'] = pd.to_datetime(df_history_old['FECHA']).dt.date

ant_cols_old = [col for col in df_history_old.columns if col.startswith('ant_')]

agg_dict_history_old = {
    'FECHA':    'first',
    'CATEGORY': lambda x: ','.join(v for v in x.dropna().unique() if v.strip() != ''),
    **{col: 'max' for col in ant_cols_old}
}

df_history_old_grouped = df_history_old.groupby('MATCH_KEY', as_index=False).agg(agg_dict_history_old)

# Rename old ant_* columns to old_hist_* so they cannot conflict with the current hist_* columns
df_history_old_grouped = df_history_old_grouped.rename(
    columns={col: col.replace('ant_', 'old_hist_', 1) for col in ant_cols_old}
)

print(f"Rows before groupby: {len(df_history_old):,}")
print(f"Rows after groupby:  {len(df_history_old_grouped):,}")
df_history_old_grouped.head()


In [ ]:
# Left join df_triaje with df_history_old_grouped on hcdm_id <-> MATCH_KEY
# Keep only old_hist_* columns from df_history_old_grouped; fill unmatched rows with 0

old_hist_cols = [col for col in df_history_old_grouped.columns if col.startswith('old_hist_')]

df_triaje = df_triaje.merge(
    df_history_old_grouped[['MATCH_KEY'] + old_hist_cols],
    how='left',
    left_on='hcdm_id',
    right_on='MATCH_KEY'
).drop(columns=['MATCH_KEY'])

# Fill 0 for unmatched rows (no MATCH_KEY found)
df_triaje[old_hist_cols] = df_triaje[old_hist_cols].fillna(0)

# has_history_old: 1 if any old_hist_* column == 1, else 0
df_triaje['has_history_old'] = (
    (df_triaje[old_hist_cols] == 1).any(axis=1).astype(int)
)

# Convert old_hist_* and has_history_old to int8
df_triaje[old_hist_cols + ['has_history_old']] = df_triaje[old_hist_cols + ['has_history_old']].astype('int8')

print(f"Rows with old history:    {df_triaje['has_history_old'].sum():,}")
print(f"Rows without old history: {(df_triaje['has_history_old'] == 0).sum():,}")
df_triaje.head()


## Active problems - Problemas (old legacy source for com_ columns)


In [ ]:
# Load old legacy active problems table
df_problems_old = pq.read_table(os.path.join(settings.source_tables_path, 
                                          "medicacion_asistencia_antecedentes/df_stroke_problemas_onehot_2026-06-26.parquet")).to_pandas()
df_problems_old.shape


In [ ]:
df_problems_old.head(5)


In [ ]:
df_problems_old.sample(5)


In [ ]:
# QC: check for duplicate hcdm_id in df_problems_old (multiple problem records per patient are expected)
dup_count_problems_old = df_problems_old['hcdm_id'].duplicated().sum()

print(f"Total rows:               {len(df_problems_old):,}")
print(f"Unique hcdm_id:           {df_problems_old['hcdm_id'].nunique():,}")
print(f"Duplicated hcdm_id rows:  {dup_count_problems_old:,} (multiple problems per patient, as expected)")
print(f"NaN in hcdm_id:           {df_problems_old['hcdm_id'].isna().sum():,}")


In [ ]:
# Group by hcdm_id: max for prob_* columns (1 if the patient has had that problem at least once)
prob_cols_old_raw = [col for col in df_problems_old.columns if col.startswith('prob_')]

df_problems_old_grouped = (
    df_problems_old
    .groupby('hcdm_id', as_index=False)[prob_cols_old_raw]
    .max()
)

# Rename old prob_* columns to old_prob_* so they cannot conflict with other problem columns
df_problems_old_grouped = df_problems_old_grouped.rename(
    columns={col: col.replace('prob_', 'old_prob_', 1) for col in prob_cols_old_raw}
)
old_prob_cols = [col for col in df_problems_old_grouped.columns if col.startswith('old_prob_')]

print(f"Rows before groupby: {len(df_problems_old):,}")
print(f"Rows after groupby:  {len(df_problems_old_grouped):,}")
df_problems_old_grouped.head()


In [ ]:
# Left join df_triaje with df_problems_old_grouped on hcdm_id
# Keep only old_prob_* columns; fill unmatched rows with 0

df_triaje = df_triaje.merge(
    df_problems_old_grouped,
    how='left',
    on='hcdm_id'
)

# Fill 0 for unmatched rows (no active problems found)
df_triaje[old_prob_cols] = df_triaje[old_prob_cols].fillna(0)

# Defragment before adding new column
df_triaje = df_triaje.copy()

# has_problems_old: 1 if any old_prob_* column == 1, else 0
df_triaje['has_problems_old'] = (
    (df_triaje[old_prob_cols] == 1).any(axis=1).astype(int)
)

# Convert old_prob_* and has_problems_old to int8
df_triaje[old_prob_cols + ['has_problems_old']] = df_triaje[old_prob_cols + ['has_problems_old']].astype('int8')

print(f"Patients with at least one old active problem: {df_triaje['has_problems_old'].sum():,}")
print(f"Patients without old active problems:          {(df_triaje['has_problems_old'] == 0).sum():,}")
df_triaje.head()


## Combination - Past history and active problems (old legacy com_ columns)


In [ ]:
# Unify old_hist_* and old_prob_* columns into com_* columns (active problems + history of disease)
# Mapping: com_name -> list of source columns to combine (max, since all are binary)
# Note: com_2_2_bedridden has no source column (no old_hist_2_2_bedridden / old_prob equivalent in the data) and is therefore omitted
com_mapping = {
    "com_1_1_hypertension": ["old_hist_1_1_hypertension", "old_prob_1_1_hypertension"],
    "com_1_2_diabetes": ["old_hist_1_2_diabetes", "old_prob_1_2_diabetes"],
    "com_1_3_dyslipidemia": ["old_hist_1_3_dyslipidemia", "old_prob_1_3_dyslipidemia"],
    "com_1_4_smoking": ["old_hist_1_4_smoking", "old_prob_1_4_smoking"],
    "com_1_5_obesity": ["old_hist_1_5_obesity", "old_prob_1_5_obesity"],
    "com_2_1_dementia": ["old_hist_2_1_dementia", "old_prob_1_18_dementia"],
    "com_3_1_ischemic_heart": ["old_hist_3_1_ischemic_heart", "old_prob_1_6_ischemic_heart"],
    "com_3_2_heart_failure": ["old_hist_3_2_heart_failure", "old_prob_1_7_heart_failure"],
    "com_3_3_arrhythmia": ["old_hist_3_3_arrhythmia", "old_prob_1_8_arrhythmia"],
    "com_3_4_peripheral_vasc": ["old_hist_3_4_peripheral_vasc", "old_prob_1_9_peripheral_vasc"],
    "com_4_1_copd": ["old_hist_4_1_copd", "old_prob_1_11_copd"],
    "com_4_2_asthma": ["old_hist_4_2_asthma", "old_prob_1_12_asthma"],
    "com_4_3_oxygen_support": ["old_hist_4_3_oxygen_support", "old_prob_1_13_oxygen_support"],
    "com_5_1_cerebrovascular": ["old_hist_5_1_cerebrovascular", "old_prob_1_10_cerebrovascular"],
    "com_5_2_epilepsy": ["old_hist_5_2_epilepsy", "old_prob_1_19_epilepsy"],
    "com_5_3_hemiplegia": ["old_hist_5_3_hemiplegia"],
    "com_6_1_renal": ["old_hist_6_1_renal", "old_prob_1_14_renal"],
    "com_6_2_hepatic": ["old_hist_6_2_hepatic", "old_prob_1_15_hepatic"],
    "com_6_3_oncologic": ["old_hist_6_3_oncologic", "old_prob_1_16_oncologic"],
    "com_6_4_connective_tissue": ["old_hist_6_4_connective_tissue", "old_prob_1_17_connective_tissue"],
    "com_6_5_osteoarticular": ["old_prob_1_20_osteoarticular"],
    "com_6_6_psychiatric": ["old_prob_1_21_psychiatric"],
    "com_6_7_urological": ["old_prob_1_22_urological"],
    "com_6_8_gi": ["old_prob_1_23_gi"],
    "com_7_1_other_medical": ["old_hist_7_other", "old_prob_1_24_other_medical"],
    "com_8_1_fall_risk": ["old_prob_2_1_fall_risk"],
    "com_8_2_self_care_deficit": ["old_prob_2_2_self_care_deficit"],
    "com_8_3_mobility_impairment": ["old_prob_2_3_mobility_impairment"],
    "com_8_4_cognitive_impairment": ["old_prob_2_4_cognitive_impairment"],
    "com_8_5_activity_intolerance": ["old_prob_2_5_activity_intolerance"],
    "com_8_6_urinary_incontinence": ["old_prob_2_6_urinary_incontinence"],
    "com_8_7_home_maintenance": ["old_prob_2_7_home_maintenance"],
    "com_8_8_therapeutic_regimen": ["old_prob_2_8_therapeutic_regimen"],
    "com_8_9_sleep_impairment": ["old_prob_2_9_sleep_impairment"],
    "com_8_10_skin_integrity_risk": ["old_prob_2_10_skin_integrity_risk"],
    "com_8_11_pain_chronic": ["old_prob_2_11_pain_chronic"],
    "com_8_12_other_nursing": ["old_prob_2_12_other_nursing"],
}

# Validate mapping sources explicitly before building com_* columns
missing_com_sources = sorted({col for cols in com_mapping.values() for col in cols if col not in df_triaje.columns})
if missing_com_sources:
    raise KeyError(f"Missing source columns for old com_* mapping: {missing_com_sources}")

# Build each com_* column as the row-wise max of its source columns (binary -> max keeps a 1 if any source is 1)
com_cols = list(com_mapping.keys())
df_triaje = df_triaje.copy()
for com_col, source_cols in com_mapping.items():
    df_triaje[com_col] = df_triaje[source_cols].max(axis=1).astype('int8')

# Drop the original old_hist_* and old_prob_* columns now that they are unified into com_*
source_cols_to_drop = [col for cols in com_mapping.values() for col in cols]
df_triaje = df_triaje.drop(columns=source_cols_to_drop)

print(f"Created {len(com_cols):,} unified com_* columns")
print(f"Dropped {len(source_cols_to_drop):,} original old_hist_*/old_prob_* columns")
df_triaje[com_cols].head()


In [ ]:
# has_com: 1 if any com_* column == 1, else 0 (old history of disease or old active problem)
df_triaje['has_com'] = (
    (df_triaje[com_cols] == 1).any(axis=1).astype(int)
)
df_triaje['has_com'] = df_triaje['has_com'].astype('int8')

print(f"Patients with com (old history or old active problem): {df_triaje['has_com'].sum():,}")
print(f"Patients without com:                                {(df_triaje['has_com'] == 0).sum():,}")

# Quality check: has_com must be the overlapping union (logical OR) of has_history_old and has_problems_old
# Method 1: derived directly from the com_* columns (computed above)
# Method 2: derived from has_history_old / has_problems_old
has_com_from_flags_old = (
    (df_triaje['has_history_old'] == 1) | (df_triaje['has_problems_old'] == 1)
).astype('int8')

mismatches_com_old = (df_triaje['has_com'] != has_com_from_flags_old).sum()
print(f"\nQC - mismatches between has_com (from com_*) and (has_history_old | has_problems_old): {mismatches_com_old:,}")
assert mismatches_com_old == 0, "has_com does not match has_history_old | has_problems_old"

df_triaje.head()


## Medication

In [ ]:
# # Load table
# df_med_ohe = pq.read_table(os.path.join(settings.source_tables_path, 
#                                           "medicacion_asistencia_antecedentes/medicacion_ohe_2026-03-20.parquet")).to_pandas()
# df_med_ohe.shape
# print(df_med_ohe.head())



In [ ]:
df_med = pq.read_table(os.path.join(settings.source_tables_path, 
                                          "medicacion_asistencia_antecedentes/medicacion_2026-03-20.parquet")).to_pandas()
df_med.shape

In [ ]:
df_med.head()

In [ ]:
df_med['nombre_grupo_atc'].value_counts()

In [ ]:
total = len(df_med['id_asistencia_fk'])
unique = df_med['id_asistencia_fk'].nunique()
duplicated = df_med['id_asistencia_fk'].duplicated().sum()

print(f"Total rows:       {total:,}")
print(f"Unique values:    {unique:,}")
print(f"Duplicated rows:  {duplicated:,}")

In [ ]:
def add_match_key(df, source_col='id_asistencia_fk', key_col='MATCH_KEY'):
    """
    Extracts the last segment of a UUID column (after the last '-') as a new MATCH_KEY column.
    Prints a comparison of unique values between the original and the new column.
    """
    df[key_col] = df[source_col].apply(lambda x: x.split('-')[-1])

    unique_original  = df[source_col].nunique()
    unique_match_key = df[key_col].nunique()

    print(f"Unique values in {source_col}: {unique_original:,}")
    print(f"Unique values in {key_col}:    {unique_match_key:,}")

    if unique_original == unique_match_key:
        print("✅ Both columns have the same number of unique values.")
    else:
        print(f"⚠️ Mismatch: {unique_original - unique_match_key:,} collisions after splitting.")

    return df

df_med = add_match_key(df_med)

In [ ]:
# One-hot encoding of name_group_atc in df_med, grouped by id_asistencia_fk
df_med_grouped = (
    pd.get_dummies(df_med[['MATCH_KEY', 'nombre_grupo_atc']], columns=['nombre_grupo_atc'])
    .groupby('MATCH_KEY', as_index=False)
    .max()
)

# Rename prefix nombre_grupo_atc_ -> atc_group_
df_med_grouped = df_med_grouped.rename(columns=lambda c: c.replace('nombre_grupo_atc_', 'atc_group_'))

# Convert columns to int8
atc_cols = [col for col in df_med_grouped.columns if col.startswith('atc_group_')]
df_med_grouped[atc_cols] = df_med_grouped[atc_cols].astype('int8')

print(f"Rows:    {len(df_med_grouped):,}")
print(f"Columns: {len(df_med_grouped.columns):,}  ({len(atc_cols)} atc_group columns)")
df_med_grouped.head()

In [ ]:
# Sanitize atc_group_* column names: replace special characters with underscores
df_med_grouped.columns = [
    re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_]', '_', col)).strip('_')
    if col.startswith('atc_group_') else col
    for col in df_med_grouped.columns
]

# Update atc_cols to reflect sanitized names
atc_cols = [col for col in df_med_grouped.columns if col.startswith('atc_group_')]

print(f"atc_group_* columns after sanitization: {len(atc_cols):,}")
print(atc_cols[:5])

In [ ]:
print(atc_cols)

In [ ]:
# Left join df_triaje with df_med_grouped on hcdm_id <-> MATCH_KEY
# Keep only atc_group_* columns from df_med_grouped; fill unmatched rows with 0

df_triaje = df_triaje.merge(
    df_med_grouped[['MATCH_KEY'] + atc_cols],
    how='left',
    left_on='hcdm_id',
    right_on='MATCH_KEY'
).drop(columns=['MATCH_KEY'])

# Fill 0 for unmatched rows (no MATCH_KEY found)
df_triaje[atc_cols] = df_triaje[atc_cols].fillna(0)

# Defragment before adding new column
df_triaje = df_triaje.copy()

# has_med: 1 if any atc_group_* column == 1, else 0
df_triaje['has_med'] = (
    (df_triaje[atc_cols] == 1).any(axis=1).astype(int)
)

# Convert atc_group_* and has_med to int8
df_triaje[atc_cols + ['has_med']] = df_triaje[atc_cols + ['has_med']].astype('int8')

print(f"Rows with medication:    {df_triaje['has_med'].sum():,}")
print(f"Rows without medication: {(df_triaje['has_med'] == 0).sum():,}")
df_triaje.head()

In [ ]:
df_triaje.head()

In [ ]:
df_triaje.shape

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)
df_triaje.info()

# Creation of new variables

## Juicio clínico - icd

In [ ]:
cols_icd = ["icd1", "icd2", "icd3"]

df_triaje[cols_icd] = (
    df_triaje[cols_icd]
    .apply(lambda col: col.astype("string").str.strip())     # strip whitespace
    .replace(r'^\s*$', pd.NA, regex=True)                    # empty strings → NA
    .replace(["nan", "NaN", "None", "none"], pd.NA)          # textual nulls → NA
    .astype("string")                                        # final enforcement
)


In [ ]:
df_triaje.head(10)

In [ ]:
def combine_clean_columns(df, columns, sep):
    """
    Clean ICD-related columns and combine valid entries into a single string.
    Missing values are returned as pd.NA.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    columns : list
        List of ICD columns to clean and combine.
    sep : str
        Separator used to join non-missing cleaned values.
    """

    def clean_value(x):
        """Normalize different missing representations to None."""
        if pd.isna(x):  # Detects real NaN/None/NaT
            return None
        if isinstance(x, str):
            x_strip = x.strip()
            # Remove empty strings and stringified missing markers
            if x_strip.lower() in ["", "nan", "none"]:
                return None
            return x_strip
        return x

    # Apply cleaning column-wise (avoid deprecated applymap)
    cleaned = df[columns].apply(lambda col: col.apply(clean_value))

    # Combine valid entries using the provided separator
    combined = cleaned.apply(
        lambda row: sep.join(str(x) for x in row if x is not None),
        axis=1
    )

    # Convert empty strings to real missing values
    combined = combined.replace("", pd.NA)

    return combined

df_triaje = df_triaje.copy()
df_triaje["icd_emerg_combined"] = combine_clean_columns(
    df_triaje,
    columns=["icd1", "icd2", "icd3"],
    sep=" | "       # You choose the separator manually
)

# Transform dtype and reorder dataframe
df_triaje = dp.transform_column_dtypes(df_triaje)


In [ ]:
df_triaje.head(10)

In [ ]:
def analyze_icd_missing_by_demand_type(df):
    """
    Analyzes missing values (pd.NA) in 'icd_emerg_combined' grouped by 'demand_type_1'.
    Provides absolute and relative values with human-readable descriptions of demand types.
    """
    
    # 1. Calculate totals per demand type to get the base for percentages
    demand_totals = df.groupby('demand_type_1').size().reset_index(name='total_cases')

    # 2. Calculate missing values in 'icd_emerg_combined' per demand type
    missing_data = df.groupby('demand_type_1').agg(
        missing_icd_count=('icd_emerg_combined', lambda x: x.isna().sum())
    ).reset_index()

    # 3. Merge both metrics
    report = pd.merge(missing_data, demand_totals, on='demand_type_1')

    # 4. Calculate the percentage of missing values within each demand category
    report['%_missing_icd'] = (report['missing_icd_count'] / report['total_cases']) * 100

    # 5. Mapping demand codes to human-readable descriptions (Integrated map)
    description_map = {
        16: "Dyspnea / respiratory distress",
        23: "Non-traumatic chest pain",
        36: "Unconsciousness / cardiac arrest",
        54: "Stroke / cerebrovascular accident",
        58: "Unconsciousness / cardiac arrest"
    }
    
    report['demand_description'] = report['demand_type_1'].map(description_map).fillna("Other / Unknown")

    # 6. Final formatting and ordering
    # We order by percentage to see which pathology is "losing" more information
    ordered_cols = [
        'demand_type_1', 'demand_description', 
        'missing_icd_count', 'total_cases', '%_missing_icd'
    ]
    
    report = report[ordered_cols].sort_values(by='%_missing_icd', ascending=False)
    report = report.round(2)

    return report

# Execution
missing_icd_report = analyze_icd_missing_by_demand_type(df_triaje)
print("=== Missing ICD-10 Analysis by Demand Type ===")
print(missing_icd_report)

In [ ]:
df_triaje = df_triaje.copy()
df_triaje["has_icd_emerg"] = df_triaje["icd_emerg_combined"].notna().astype("int8")
print(df_triaje["has_icd_emerg"].value_counts())

In [ ]:
df_triaje['icd_emerg_combined'].isna().sum()

In [ ]:
dp.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
df_triaje.loc[
    df_triaje["icd_emerg_combined"].str.contains("798", na=False),
    "resolution_code_id"
].value_counts(dropna=False)


In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(10)

## P1 assigned

In [ ]:
print("Value counts of 'priority_assigned' column before conversion:")
df_triaje['priority_assigned'].value_counts(dropna = False)

In [ ]:
print(df_triaje['priority_assigned'].dtype)

In [ ]:
# priority_assigned is the priority predicted during the call, assigned by the system
# Creating p1_assigned: 1 is maximum priority, 0 is non-maximum priority
df_triaje['p1_assigned'] = (df_triaje['priority_assigned'] == 1).astype('int8')

In [ ]:
print("Value counts of 'p1_assigned' column after conversion:")
df_triaje['p1_assigned'].value_counts(dropna = False)

In [ ]:
print(df_triaje['p1_assigned'].dtype)

## P1 Real Emergencies

In [ ]:
# Load table linking ICD to P1, together with special cases containing specific triage questions
icd_table = pd.read_csv(
    os.path.join(settings.source_tables_path, "icd/icd_p1outcomes_20251218.tsv"),
    dtype=str,
    encoding="utf-8",
    sep="\t"
)


In [ ]:
icd_table.head(10)

In [ ]:
icd_table["AND/Exclusion"].value_counts(dropna=False)


In [ ]:
icd_table.info()

In [ ]:
print(len(icd_table))
icd_table["ICD_original"].nunique()

In [ ]:
def process_icd_codes(df, icd_combined_col, reference_table, reference_icd_col, p1_col, drop_excluded=True):
    """
    Process combined ICD codes to detect ICD exclusions, P1 priority flags,
    and apply additional specific triage rules based on AND/Exclusion categories.

    Parameters
    ----------
    df : pandas.DataFrame
        Main dataset containing the combined ICD codes column.
    icd_combined_col : str
        Name of the column containing combined ICD codes separated by "|".
    reference_table : pandas.DataFrame
        ICD reference table containing ICD codes, P1 labels, and Exclusion labels.
    reference_icd_col : str
        Name of the ICD column inside the reference table (e.g., 'ICD_original').
    p1_col : str
        Name of the P1 priority column to create (e.g. 'p1_real_emerg', 'p1_real_bps').
    drop_excluded : bool, default True
        If True, rows flagged by the exclusion criteria are removed from the
        dataframe. If False, those rows are kept with <NA> in p1_col.

    Returns
    -------
    df_filtered : pandas.DataFrame
        Dataframe with a new column named after p1_col.
    n_rows_excluded_icd : int
        Number of rows flagged by the exclusion criteria (removed only if drop_excluded=True).
    """

    # ===============================================================
    # 1. Normalize ICD reference table and create lookup dictionaries
    # ===============================================================
    ref = reference_table.copy()

    ref[reference_icd_col] = ref[reference_icd_col].astype("string").str.strip()

    ref["p1_flag"] = ref["P1 outcomes"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().upper() == "P1" else 0
    )
    ref["exclusion_flag"] = ref["AND/Exclusion"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().lower() == "exclusion" else 0
    )

    p1_map = dict(zip(ref[reference_icd_col], ref["p1_flag"]))
    exclusion_map = dict(zip(ref[reference_icd_col], ref["exclusion_flag"]))

    # ===============================================================
    # 2. Function to evaluate a single row of combined ICD codes
    # ===============================================================
    def evaluate_icd_row(text):
        if pd.isna(text):
            # Maintain None for rows without ICD information
            return {"exclude": False, "p1": None}

        raw_codes = [c.strip() for c in str(text).split("|")]
        codes = [c for c in raw_codes if c != ""]

        for code in codes:
            if code in exclusion_map and exclusion_map[code] == 1:
                return {"exclude": True, "p1": None}

        for code in codes:
            if code in p1_map and p1_map[code] == 1:
                return {"exclude": False, "p1": 1}

        return {"exclude": False, "p1": 0}


    # ===============================================================
    # 3. Apply evaluation row-wise across the dataset
    # ===============================================================
    evaluation = df[icd_combined_col].apply(evaluate_icd_row)

    df[p1_col] = evaluation.apply(lambda r: r["p1"])
    df["exclude_row"] = evaluation.apply(lambda r: r["exclude"])

    n_rows_excluded_icd = df["exclude_row"].sum()

    # ===============================================================
    # 4. Remove excluded rows (only if drop_excluded=True)
    # ===============================================================
    n_rows_previous = len(df)

    if drop_excluded:
        df_filtered = df[df["exclude_row"] == False].drop(columns=["exclude_row"]).copy()
        df_filtered = df_filtered.reset_index(drop=True)
    else:
        df_filtered = df.drop(columns=["exclude_row"]).copy()

    n_rows_final = len(df_filtered)

    # ===============================================================
    # 5. Apply specific triage rules based on AND/Exclusion categories
    # ===============================================================

    # Identify ICD groups associated with special rules
    icd_speech = ref.loc[
        ref["AND/Exclusion"] == "Habla con dificultad", 
        reference_icd_col
    ].dropna().astype(str).tolist()

    icd_normal = ref.loc[
        ref["AND/Exclusion"] == "Vida Normal y/o limitada, pero se vale por si mismo",
        reference_icd_col
    ].dropna().astype(str).tolist()

    # Split ICDs into lists per row
    df_filtered["icd_list"] = (
        df_filtered[icd_combined_col]
        .astype("string")
        .fillna("<NA>")
        .apply(lambda x: [c.strip() for c in x.split("|")] if x != "<NA>" else [])
    )

    # Helper function for matching
    def row_matches_any(row_icds, target_icds):
        return any(c in target_icds for c in row_icds)

    # Define common masks for rule application
    p1_before = df_filtered[p1_col].copy()
    mask_has_icd = df_filtered[p1_col].notna()

    # --- Rule set A: "Habla con dificultad"
    mask_speech_icd = df_filtered["icd_list"].apply(lambda lst: row_matches_any(lst, icd_speech))

    mask_speech_true = (
        ((df_filtered["demand_type_1"] == 16) & (df_filtered["q3"] == "b")) |
        ((df_filtered["demand_type_1"] == 54) & (df_filtered["q3"] == "d"))
    )

    # Apply rule A ensuring NA values are preserved
    df_filtered.loc[mask_speech_icd & mask_speech_true & mask_has_icd, p1_col] = 1
    df_filtered.loc[mask_speech_icd & ~mask_speech_true & mask_has_icd, p1_col] = 0


    # --- Rule set B: "Vida Normal y/o limitada, pero se vale por si mismo"
    mask_normal_icd = df_filtered["icd_list"].apply(lambda lst: row_matches_any(lst, icd_normal))

    mask_normal_true = (
        (df_filtered["demand_type_1"] == 54) & (df_filtered["q6"].isin(["a", "0"]))
    )

    # Apply rule B ensuring NA values are preserved
    df_filtered.loc[mask_normal_icd & mask_normal_true & mask_has_icd, p1_col] = 1
    df_filtered.loc[mask_normal_icd & ~mask_normal_true & mask_has_icd, p1_col] = 0

    # Count how many rows were really changed by the specific rules.
    # Missing values compare as different in pandas, so restrict the count
    # to rows that had an assignable ICD-derived P1 before these rules.
    n_p1_assignable_before_specific_rules = mask_has_icd.sum()
    n_p1_updated_specific_rules = ((df_filtered[p1_col] != p1_before) & mask_has_icd).sum()

    # Remove helper column
    df_filtered = df_filtered.drop(columns=["icd_list"])

    # ===============================================================
    # 6. Return dataframe and statistics
    # ===============================================================
    stats = {
        "n_rows_previous": n_rows_previous,
        "n_rows_final": n_rows_final,
        "n_rows_excluded_icd": n_rows_excluded_icd,
        "n_p1_assignable_before_specific_rules": n_p1_assignable_before_specific_rules,
        "n_p1_updated_specific_rules": n_p1_updated_specific_rules,
    }

    # ===============================================================
    # 7. Print summary and finalize data types
    # ===============================================================
    print(f"\n================ ICD Processing Summary ({p1_col}) ================\n")
    print(f"✔ Total rows before applying ICD exclusion criteria: {stats['n_rows_previous']:,}")
    if drop_excluded:
        print(f"✔ Total rows removed due to ICD exclusion rules: {stats['n_rows_excluded_icd']:,}")
        print(f"✔ Total rows remaining after applying ICD exclusion criteria: {stats['n_rows_final']:,}")
    else:
        print(f"✔ Total rows flagged by ICD exclusion rules (kept with <NA>): {stats['n_rows_excluded_icd']:,}")
    print(f"✔ Rows with assignable ICD-derived P1 before specific triage rules: {stats['n_p1_assignable_before_specific_rules']:,}")
    print(f"✔ Rows where P1 was reassigned due to specific triage rules: {stats['n_p1_updated_specific_rules']:,}")
    print(f"✔ Rows with <NA> in {p1_col} (no ICD data or excluded): {df_filtered[p1_col].isna().sum():,}")
    print("\n========================================================\n")

    # Final conversion to nullable Int8 for memory efficiency and NA support
    df_filtered[p1_col] = df_filtered[p1_col].astype("Int8")

    return df_filtered.reset_index(drop=True), n_rows_excluded_icd

# Call the function
df_triaje, n_rows_excluded_icd = process_icd_codes(
    df=df_triaje,
    icd_combined_col="icd_emerg_combined",
    reference_table=icd_table,
    reference_icd_col="ICD_original",
    p1_col="p1_real_emerg",
)

In [ ]:
df_triaje["p1_real_emerg"].value_counts(dropna=False)

In [ ]:
def downgrade_p1_for_palliative_icd(
    df,
    icd_col,
    patterns,
    target_col
):
    """
    Downgrade P1 priority to 0 for rows where the combined ICD column contains
    any ICD associated with palliative care. Only counts rows whose P1 changes
    from 1 → 0.

    The function parses the combined ICD column safely by splitting on '|'
    to avoid false substring matches (e.g., matching '50' inside '150').

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    icd_col : str
        Name of the column with combined ICD codes separated by '|'.
    patterns : list of str
        ICD codes indicating palliative-care situations (exact match).
    target_col : str, default="p1_real"
        Name of the P1 priority variable to update.

    Returns
    -------
    df : pandas.DataFrame
        Updated dataframe.
    n_changed : int
        Number of rows whose P1 priority changed from 1 → 0.
    """

    # Normalize ICD patterns
    patterns = [p.lower().strip() for p in patterns]

    print("\n================ Palliative ICD Detection Summary ================\n")

    # ------------------------------------------------------------------
    # 1. Extract ICD lists per row by splitting icd_col on '|'
    # ------------------------------------------------------------------
    df["_icd_list"] = (
        df[icd_col]
        .astype("string")
        .str.lower()
        .fillna("<NA>")
        .apply(lambda x: [c.strip() for c in x.split("|")] if x != "<NA>" else [])
    )

    # ------------------------------------------------------------------
    # 2. Count rows containing each ICD pattern separately (exact match)
    # ------------------------------------------------------------------
    for p in patterns:
        mask_p = df["_icd_list"].apply(lambda codes: p in codes)
        print(f"✔ Rows containing ICD '{p}': {mask_p.sum():,}")

    # ------------------------------------------------------------------
    # 3. Combined mask: any ICD from patterns found in the row
    # ------------------------------------------------------------------
    mask = df["_icd_list"].apply(lambda codes: any(p in codes for p in patterns))

    # Identify rows whose P1 will change
    p1_before = df[target_col].copy()
    change_mask = mask & (p1_before == 1)

    # ------------------------------------------------------------------
    # 4. Apply the downgrading rule: set P1 = 0
    # ------------------------------------------------------------------
    df.loc[mask, target_col] = 0

    # Number of rows changed from P1=1 to P1=0
    n_changed = change_mask.sum()

    print("\n================ Palliative ICD Rule Application ================\n")
    print(f"✔ Patterns searched (palliative ICDs): {patterns}")
    print(f"✔ Rows downgraded from P1 → 0 due to palliative ICDs: {n_changed:,}")
    print("\n=================================================================\n")

    # Clean helper column
    df = df.drop(columns=["_icd_list"])

    return df

df_triaje = downgrade_p1_for_palliative_icd(
    df=df_triaje,
    icd_col="icd_emerg_combined",
    patterns=["z51.5", "v66.7"],  # palliative ICDs
    target_col="p1_real_emerg"
)



In [ ]:
df_triaje["p1_real_emerg"].value_counts(dropna=False)

In [ ]:
import pandas as pd

def update_p1_by_death_resolution(
        df, 
        target_col,
        icd_col="icd_emerg_combined",
        resolution_col="resolution_code_id",
        death_codes=None
    ):
    """
    Assigns target_col = 1 for cases where the ICD is missing AND 
    the resolution code indicates a death outcome.
    """
    
    # -----------------------------------------------------------
    # 1. Default death resolution code sets
    # -----------------------------------------------------------
    if death_codes is None:
        death_codes = ["32", "33"]

    # -----------------------------------------------------------
    # 2. Identify missing ICDs and extract resolution codes
    # -----------------------------------------------------------
    # We look for rows that currently have <NA> in the target column
    icd_missing_mask = df[target_col].isna()

    # Process resolution codes (handling potential '|' separators)
    res_expanded = (
        df[resolution_col]
        .astype("string")
        .str.split("|")
        .explode()
        .str.strip()
    )

    # -----------------------------------------------------------
    # 3. Identify rows with death codes
    # -----------------------------------------------------------
    has_death = res_expanded.isin(death_codes).groupby(level=0).any()

    # Ensure alignment with original dataframe
    has_death = has_death.reindex(df.index, fill_value=False)

    # -----------------------------------------------------------
    # 4. Apply rule: target IS <NA> AND death → target = 1
    # -----------------------------------------------------------
    rule_mask = icd_missing_mask & has_death
    
    # Calculation of stats before the update to avoid NA comparison issues
    n_updated = rule_mask.sum()

    # Update only the missing values that meet the criteria
    df.loc[rule_mask, target_col] = 1

    print("\n================ Death Resolution Rules Summary ================\n")
    print(f"✔ Rows updated from <NA> to P1 (Death criteria): {n_updated:,}")
    print("================================================================\n")

    # Ensure final type is nullable Int8
    df[target_col] = df[target_col].astype("Int8")

    return df, n_updated

# Execution
df_triaje, n_updated_death = update_p1_by_death_resolution(
    df_triaje, 
    target_col="p1_real_emerg"
)

In [ ]:
df_triaje["p1_real_emerg"].value_counts(dropna=False)

In [ ]:
df_triaje[df_triaje["icd_emerg_combined"].notna()].head(5)

In [ ]:
def analyze_p1_missing_by_demand_type(df):
    """
    Analyzes missing values (pd.NA), positive cases (1) and negative cases (0) 
    in 'p1_real_emerg' grouped by 'demand_type_1'. Provides absolute and relative values.
    """
    
    # 1. Calculate totals per demand type
    demand_totals = df.groupby('demand_type_1').size().reset_index(name='total_cases')

    # 2. Calculate missing values, P1 (1) and non-P1 (0) counts
    # We use lambda functions to handle the nullable Int8/Int64 types correctly
    metrics_data = df.groupby('demand_type_1').agg(
        p1_count=('p1_real_emerg', lambda x: (x == 1).sum()),
        p0_count=('p1_real_emerg', lambda x: (x == 0).sum()),
        missing_p1_count=('p1_real_emerg', lambda x: x.isna().sum())
    ).reset_index()

    # 3. Merge metrics
    report = pd.merge(metrics_data, demand_totals, on='demand_type_1')

    # 4. Calculate the percentage of missing target values
    report['%_missing_p1'] = (report['missing_p1_count'] / report['total_cases']) * 100

    # 5. Mapping demand codes to human-readable descriptions
    description_map = {
        16: "Dyspnea / respiratory distress",
        23: "Non-traumatic chest pain",
        36: "Unconsciousness / cardiac arrest",
        54: "Stroke / cerebrovascular accident",
        58: "Unconsciousness / cardiac arrest"
    }
    
    report['demand_description'] = report['demand_type_1'].map(description_map).fillna("Other / Unknown")

    # 6. Final formatting and ordering
    # Columns ordered for a clear view of Data Quality and Class Balance
    ordered_cols = [
        'demand_type_1', 'demand_description', 
        'p1_count', 'p0_count', 'missing_p1_count', 
        'total_cases', '%_missing_p1'
    ]
    
    report = report[ordered_cols].sort_values(by='%_missing_p1', ascending=False)
    report = report.round(2)

    return report

# Execution example
target_analysis_report = analyze_p1_missing_by_demand_type(df_triaje)
print("=== Target (p1_real_emerg) Analysis by Demand Type ===")
print(target_analysis_report)

## P1 Real Hospital (BPS)

In [ ]:
# Call the function for the BPS (hospital) ICD source: keep all rows (no dropping)
# and reuse the same speech/normal-life triage override rules
df_triaje, n_rows_excluded_icd_bps = process_icd_codes(
    df=df_triaje,
    icd_combined_col="icd_codes_bps",
    reference_table=icd_table,
    reference_icd_col="ICD_original",
    p1_col="p1_real_bps",
    drop_excluded=False,
)

# Downgrade P1 to 0 for palliative-care ICDs
df_triaje = downgrade_p1_for_palliative_icd(
    df=df_triaje,
    icd_col="icd_codes_bps",
    patterns=["z51.5", "v66.7"],  # palliative ICDs
    target_col="p1_real_bps"
)

In [ ]:
df_triaje["p1_real_bps"].value_counts(dropna=False)

In [ ]:
df_triaje.info()

## P1 Real Emerg & Hosp

In [ ]:
# Combine p1_real_emerg and p1_real_bps: when both are valid (0 or 1),
# p1_real_emerg takes priority; otherwise fall back to whichever is valid
df_triaje["p1_real_emerg_bps"] = df_triaje["p1_real_emerg"].fillna(df_triaje["p1_real_bps"])

In [ ]:
df_triaje["p1_real_emerg_bps"].value_counts(dropna=False)

In [ ]:
# Rows where p1_real_emerg is NA but p1_real_bps is valid (0 or 1)
n_only_bps_valid = (df_triaje["p1_real_emerg"].isna() & df_triaje["p1_real_bps"].notna()).sum()
print(f"Rows with NA in p1_real_emerg but valid in p1_real_bps: {n_only_bps_valid:,}")

In [ ]:
# Among rows where both p1_real_emerg and p1_real_bps are valid (0 or 1),
# count how many conflicting BPS values are overridden by p1_real_emerg
mask_both_valid = df_triaje["p1_real_emerg"].notna() & df_triaje["p1_real_bps"].notna()
n_bps_overridden_by_emerg = (df_triaje.loc[mask_both_valid, "p1_real_emerg"] != df_triaje.loc[mask_both_valid, "p1_real_bps"]).sum()
print(f"Rows with both p1_real_emerg and p1_real_bps valid: {mask_both_valid.sum():,}")
print(f"Rows where p1_real_bps was overridden by p1_real_emerg: {n_bps_overridden_by_emerg:,}")

### Inspect patients without P1 in emergencies and CR 35-36

In [ ]:
resolution_codes_35_36 = {"35", "36"}

resolution_code_parts = (
    df_triaje["resolution_code_id"]
    .astype("string")
    .str.split("|")
    .explode()
    .str.strip()
)

has_resolution_35_36 = (
    resolution_code_parts.isin(resolution_codes_35_36)
    .groupby(level=0)
    .any()
    .reindex(df_triaje.index, fill_value=False)
)

mask_p1_real_emerg_na_resolution_35_36 = df_triaje["p1_real_emerg"].isna() & has_resolution_35_36
n_p1_real_emerg_na_resolution_35_36 = mask_p1_real_emerg_na_resolution_35_36.sum()

print(
    "Rows with NA in p1_real_emerg and resolution_code_id containing 35 or 36: "
    f"{n_p1_real_emerg_na_resolution_35_36:,}"
)

mask_healthcard_or_national_na = df_triaje["healthcard_id"].isna() | df_triaje["national_id"].isna()

n_p1_real_emerg_na_resolution_35_36_healthcard_or_national_na = (
    mask_p1_real_emerg_na_resolution_35_36 & mask_healthcard_or_national_na
).sum()

print(
    "Rows with NA in p1_real_emerg and resolution_code_id containing 35 or 36 "
    "with NA in "
    "healthcard_id or national_id: "
    f"{n_p1_real_emerg_na_resolution_35_36_healthcard_or_national_na:,}"
)

In [ ]:
# Step 1: base (already computed above)
n_base = n_p1_real_emerg_na_resolution_35_36
print(f'Base (NA p1_real_emerg + CR 35/36):              {n_base:,}')

# Step 2: remove rows with NA patient ID
n_removed_no_id = n_p1_real_emerg_na_resolution_35_36_healthcard_or_national_na
mask_step2 = mask_p1_real_emerg_na_resolution_35_36 & ~mask_healthcard_or_national_na
n_step2 = mask_step2.sum()
print(f'Removed (NA healthcard_id or national_id):        {n_removed_no_id:,}')
print(f'After step 2:                                     {n_step2:,}')
assert n_step2 == n_base - n_removed_no_id, f'Step 2 mismatch: {n_step2} != {n_base} - {n_removed_no_id}'

# Step 3: remove rows already resolved via p1_real_bps (hospital label available)
mask_bps_valid = df_triaje['p1_real_bps'].notna()
n_removed_bps = (mask_step2 & mask_bps_valid).sum()
df_cr3536_valid = df_triaje[mask_step2 & ~mask_bps_valid].copy()
n_final = len(df_cr3536_valid)
print(f'Removed (p1_real_bps not NA):                     {n_removed_bps:,}')
print(f'Final subset:                                      {n_final:,}')
assert n_final == n_step2 - n_removed_bps, f'Step 3 mismatch: {n_final} != {n_step2} - {n_removed_bps}'
print('All checks passed.')

In [ ]:
df_cr3536_valid['age'].describe()

In [ ]:
df_cr3536_valid['sex'].value_counts(dropna=False)

In [ ]:
df_cr3536_valid['resolution_code_id'].value_counts(dropna=False)

In [ ]:
df_cr3536_valid['resource_type'].value_counts(dropna=False)

## triage

In [ ]:
# Create the 'triage' variable:
# - triage = 0 → all demand-specific question fields (q1–q7) are equal to "0"
#                meaning no demand-specific information was collected during the call.
# - triage = 1 → at least one demand-specific question has a value different from "0",
#                indicating that structured triage information was captured.


question_cols = ["q1","q2","q3","q4","q5","q6","q7"]

# Compute triage flag: 1 if any values != "0", otherwise 0
df_triaje["triage"] = (
    (df_triaje[question_cols] != "0").any(axis=1).astype("int8")
)

# Convert to categorical dtype
df_triaje["triage"] = df_triaje["triage"].astype("category")


## Date/Time variables

In [ ]:
# Create the day of the week (number, Monday=0, Sunday=6) variable
df_triaje['day_week'] = df_triaje['demand_date'].dt.dayofweek

In [ ]:
df_triaje['day_week'].value_counts(dropna = False)

In [ ]:
# Create the month (number) variable
df_triaje['month'] = df_triaje['demand_date'].dt.month

In [ ]:
df_triaje['month'].value_counts(dropna = False)

In [ ]:
# Create the year variable
df_triaje['year'] = df_triaje['demand_date'].dt.year

In [ ]:
df_triaje['year'].value_counts(dropna = False)

In [ ]:
# Display the new columns and the original date column
print(df_triaje[['demand_date', 'day_week', 'month', 'year']].head())

In [ ]:
def add_season_variable(df, date_col, output_col="season"):
    """
    Add a 'season' numerical column to the dataframe based on a date column.
    Seasons are encoded as integers to preserve ordinality and ensure 
    compatibility with ML models (avoiding categorical mismatch errors).
    
    Mapping:
        - 0: Spring (21/03–20/06)
        - 1: Summer (21/06–22/09)
        - 2: Autumn (23/09–20/12)
        - 3: Winter (rest of the dates)

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    date_col : str
        Name of the datetime column used to compute the season.
    output_col : str, default="season"
        Name of the resulting season column.
    """

    # Compute MMDD as a single integer (e.g., March 21 -> 321)
    # Using float temporarily to handle potential NaNs safely
    md = (df[date_col].dt.month * 100 + df[date_col].dt.day)

    # Define seasonal conditions based on Northern Hemisphere cutoff dates
    conditions = [
        md.between(321, 620, inclusive="both"),   # Spring
        md.between(621, 922, inclusive="both"),   # Summer
        md.between(923, 1220, inclusive="both")   # Autumn
    ]
    
    # Define numerical choices (0 to 2). Winter will be the default (3).
    choices = [0, 1, 2]

    # Assign seasonal values using fast NumPy selection
    # Default is 3 (Winter)
    df[output_col] = np.select(conditions, choices, default=3)

    # Convert to int8 for memory efficiency and ensure compatibility
    # If the date is NaT, the result of md is NaN, and we should preserve it as -1
    df[output_col] = df[output_col].astype(np.int8)
    df.loc[df[date_col].isna(), output_col] = -1

    return df

# Usage
df_triaje = add_season_variable(df_triaje, "demand_date", output_col="season")

In [ ]:
df_triaje["season"].value_counts(dropna=False)

In [ ]:

def add_time_slot(df, primary_date_col, backup_date_col, output_col="time_of_day"):
    """
    Create a numerical time-of-day slot from a primary datetime column, 
    falling back to a backup column if the primary is NaT. Rows where both 
    columns are NaT are excluded.

    Mapping (Hours 0–23):
        - 0: early_morning (00–05)
        - 1: morning       (06–11)
        - 2: afternoon     (12–17)
        - 3: night         (18–23)
    """
    initial_rows = len(df)

    # 1. Create a combined hour series (Coalesce logic)
    # If primary is NaT, use backup. If both are NaT, it remains NaN.
    combined_hour = df[primary_date_col].dt.hour.fillna(df[backup_date_col].dt.hour)

    # 2. Drop rows where both date columns are NaT
    df = df.dropna(subset=[primary_date_col, backup_date_col], how='all').copy()
    
    final_rows = len(df)
    n_rows_excluded_time = initial_rows - final_rows

    # 3. Apply numerical binning to the hours of the remaining rows
    # We re-calculate combined_hour for the filtered dataframe
    hour_filtered = df[primary_date_col].dt.hour.fillna(df[backup_date_col].dt.hour)
    
    df[output_col] = pd.cut(
        hour_filtered,
        bins=[-1, 5, 11, 17, 23],
        labels=[0, 1, 2, 3],
        right=True
    ).astype(np.int8)

    # 4. Reporting
    print(f"=== Time Slot Processing ===")
    print(f"Original rows: {initial_rows:,}")
    print(f"Excluded rows (both dates NaT): {n_rows_excluded_time:,}")
    print(f"Final rows: {final_rows:,}")
    print(f"============================\n")

    return df, n_rows_excluded_time

# Usage
df_triaje, n_rows_excluded_time = add_time_slot(
    df_triaje, 
    primary_date_col="contact_entry_time", 
    backup_date_col="activation_time", 
    output_col="time_of_day"
)

In [ ]:
df_triaje["time_of_day"].value_counts(dropna=False)

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
import pandas as pd

# 1. Filtrar las filas donde resolution_code_id es igual a 31
df_triaje_test = df_triaje[df_triaje['resolution_code_id'] == 31]

# 2. Agrupar por 'demand_type' y contar las ocurrencias
resultado = df_triaje_test.groupby('demand_type_1').size().reset_index(name='cantidad_filas')

print(resultado)

In [ ]:
df_triaje.sample(20)

# Final exclusion criteria

## Resolucion code: found death (31)

In [ ]:
# EDA: exploration of data

def analyze_resolution_codes_31(df):
    """
    Analyzes resolution codes within the triage dataset, specifically focusing on code '31'
    and tracking missing ICD information across different demand types.
    """
    
    # Calculate original totals per demand type
    totals_per_type = df.groupby('demand_type_1').size().reset_index(name='total_original')

    # Filter rows containing '31' and create a subset for detailed analysis
    df_res_31 = df[df['resolution_code_id'].astype(str).str.contains('31', na=False)].copy()

    # Calculate metrics for count of '31' and missing ICD entries
    # Counts total rows for '31' and rows where icd_emerg_combined is null
    analysis_results = df_res_31.groupby('demand_type_1').agg(
        count_31=('demand_type_1', 'size'),
        icd_missing_in_31=('icd_emerg_combined', lambda x: x.isna().sum())
    ).reset_index()

    # Merge with original totals and calculate percentages
    final_report = pd.merge(analysis_results, totals_per_type, on='demand_type_1')

    # Percentage of '31' cases over the original total for that demand type
    final_report['%_31_total'] = (final_report['count_31'] / final_report['total_original']) * 100

    # Percentage of missing ICD entries specifically within the '31' group
    final_report['%_icd_missing_in_31'] = (final_report['icd_missing_in_31'] / final_report['count_31']) * 100

    # Map technical codes to human-readable descriptions
    description_map = {
        16: "Dyspnea / respiratory distress",
        23: "Non-traumatic chest pain",
        36: "Unconsciousness / cardiac arrest",
        54: "Stroke / cerebrovascular accident",
        58: "Unconsciousness / cardiac arrest"
    }
    final_report['description'] = final_report['demand_type_1'].map(description_map)

    # Final formatting: reorder columns and round numerical values to 2 decimals
    ordered_cols = [
        'demand_type_1', 'description', 'count_31', 'total_original', 
        '%_31_total', 'icd_missing_in_31', '%_icd_missing_in_31'
    ]
    
    final_report = final_report[ordered_cols].sort_values(by='description')
    final_report = final_report.round(2)

    return final_report

# Call the function
resolution_analysis_df = analyze_resolution_codes_31(df_triaje)
print(resolution_analysis_df)

In [ ]:
# Exclusion of rows with missing ICD information in resolution code '31' (found death)

def exclude_missing_icd_in_code31(df):
    """
    Excludes rows that have resolution code '31' (handling pipe-separated values)
    and missing ICD information, with robust null handling.
    """
    def contains_code_31(val):
        # pd.isna handles None, NaN, and <NA>
        if pd.isna(val):
            return False
        
        # Convert to string and handle potential stringified nulls
        val_str = str(val).strip().lower()
        if val_str in ['none', 'nan', '']:
            return False
            
        # Split by '|', strip whitespace from each element, and check for '31'
        codes = [code.strip() for code in val_str.split('|')]
        return '31' in codes

    # Identify the rows to be excluded
    # We use .isna() for icd_emerg_combined as it is the standard for missing data
    mask_to_exclude = (
        df['resolution_code_id'].apply(contains_code_31) & 
        df['icd_emerg_combined'].isna()
    )
    
    # Calculate counts for reporting
    initial_rows = len(df)
    n_rows_excluded_code31 = mask_to_exclude.sum()
    
    # Perform exclusion
    df_cleaned = df[~mask_to_exclude].copy()
    final_rows = len(df_cleaned)
    
    # Print results for audit trail
    print(f"Rows before exclusion: {initial_rows:,}")
    print(f"Rows excluded (Exact Code 31 with missing ICD): {n_rows_excluded_code31:,}")
    print(f"Rows after exclusion: {final_rows:,}")
    
    return df_cleaned, n_rows_excluded_code31

# Execution:
df_triaje, n_rows_excluded_code31 = exclude_missing_icd_in_code31(df_triaje)

## P1 real: exclusion of patients without ICD in emergency or hospital

In [ ]:
df_triaje['p1_real_emerg'].value_counts(dropna = False)
df_triaje['p1_real_bps'].value_counts(dropna = False)
df_triaje['p1_real_emerg_bps'].value_counts(dropna = False)

In [ ]:
# Exclusion of rows without ICD information during emergency or hospital assistance
n_rows_before_p1real_exclusion = len(df_triaje)
print(f"Number of rows before filtering out missing p1_real_emerg: {n_rows_before_p1real_exclusion:,}")

# Filter out rows where there are no ICD during emergency assistances
df_triaje = df_triaje[df_triaje["p1_real_emerg_bps"].notna()]

n_rows_after_p1real_exclusion = len(df_triaje)
print(f"Number of rows after filtering out missing p1_real_emerg: {n_rows_after_p1real_exclusion:,}")

n_rows_excluded_p1real = n_rows_before_p1real_exclusion - n_rows_after_p1real_exclusion
print(
    "Rows removed after filtering out missing p1_real_emerg_bps:",
    n_rows_excluded_p1real
)


In [ ]:
df_triaje['p1_real_emerg'].value_counts(dropna = False)
df_triaje['p1_real_bps'].value_counts(dropna = False)
df_triaje['p1_real_emerg_bps'].value_counts(dropna = False)

# Data preparation for modeling

## triaje answers - one-hot encoding

In [ ]:

def encode_triage_questions(df, question_cols):
    """
    Performs custom One-Hot Encoding for triage questions with robust dtype handling.
    Values: 1 (selected), 0 (not selected), -1 (question not asked).
    """
    df_encoded = df.copy()
    
    for col in question_cols:
        # Identify rows where the question was NOT asked (original value "0")
        not_asked_mask = df_encoded[col].astype(str) == "0"
        
        # Get standard One-Hot Encoding. 
        # dtype=int prevents the boolean-to-integer FutureWarning.
        dummies = pd.get_dummies(df_encoded[col], prefix=col, dtype=int)
        
        # Drop the '0' column as it represents the "not asked" state which we handle with -1
        col_zero = f"{col}_0"
        if col_zero in dummies.columns:
            dummies = dummies.drop(columns=[col_zero])
        
        # Now we can safely assign -1 because the dtype is already integer
        dummies.loc[not_asked_mask, :] = -1
        
        # Add to the main dataframe and remove original column
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        df_encoded = df_encoded.drop(columns=[col])
        
    return df_encoded

# Execute encoding
df_triaje = encode_triage_questions(df_triaje, question_cols)

## Categorical variables - One-hot encoding

In [ ]:
# ---------------------------------------------------------------------------
# One-hot encoding – categorical variables with known label mappings
# Each original numeric column is expanded into binary dummy columns named
# <variable>_<label>. Original columns are kept unchanged.
# ---------------------------------------------------------------------------

def add_dummies(df, col, code_map):
    for code, label in code_map.items():
        df[f"{col}_{label}"] = (df[col] == code).astype("int8")
    return df

df_triaje = add_dummies(df_triaje, "day_week", {
    0: "monday", 1: "tuesday", 2: "wednesday", 3: "thursday",
    4: "friday",  5: "saturday", 6: "sunday",
})

df_triaje = add_dummies(df_triaje, "time_of_day", {
    0: "early_morning", 1: "morning", 2: "afternoon", 3: "night",
})

df_triaje = add_dummies(df_triaje, "month", {
    1: "january",   2: "february",  3: "march",    4: "april",
    5: "may",       6: "june",      7: "july",     8: "august",
    9: "september", 10: "october", 11: "november", 12: "december",
})

df_triaje = add_dummies(df_triaje, "season", {
    0: "spring", 1: "summer", 2: "autumn", 3: "winter",
})

df_triaje = add_dummies(df_triaje, "location_patient", {
    0: "home", 1: "public_road", 2: "other",
})

df_triaje = add_dummies(df_triaje, "alert_receiver", {
    0: "112", 1: "user", 2: "pol_fg", 3: "hs", 4: "tele", 5: "others",
})

df_triaje = add_dummies(df_triaje, "province", {
    0: "almeria", 1: "cadiz", 2: "cordoba", 3: "granada",
    4: "huelva",  5: "jaen",    6: "malaga",  7: "sevilla",
})

dummy_cols = [c for c in df_triaje.columns if any(
    c.startswith(v + "_") for v in
    ["day_week", "time_of_day", "month", "season", "location_patient", "alert_receiver", "province"]
)]
print(f"Added {len(dummy_cols):,} dummy columns:")
print(dummy_cols)

## Free text - SBERT embedding

In [ ]:

def get_text_embeddings(df, column_name='literal_reason', batch_size=512):
    """
    Generate semantic text embeddings for medical descriptions using SBERT.
    The resulting dataframe includes only 'demandpk', the original text column, 
    and the new embeddings.
    
    This function saves the result to a Parquet file if settings.run_embedding_sbert is enabled.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing 'demandpk' and the text column.
    column_name : str, default='literal_reason'
        Name of the column to encode.
    batch_size : int, default=128
        Number of sentences processed at once (uses GPU if available).

    Returns
    -------
    pd.DataFrame or None
        A filtered dataframe containing [demandpk, column_name, emb_0...emb_n]
        if the process is enabled, otherwise returns None.
    """
    
    # Check the run_embedding_sbert setting to handle specific cases
    if settings.run_embedding_sbert == 0:
        print("--- Skipping embedding generation as run_embedding_sbert is set to 0 ---")
        return None
    elif settings.run_embedding_sbert == 1:
        # Proceed with embedding generation
        pass
    else:
        # Print error if the value is neither 0 nor 1
        print("Error: settings.run_embedding_sbert must be either 0 or 1.")
        return None
    
    print(f"--- Generating embeddings for {len(df):,} rows in column '{column_name}' ---")
    
    # Load a lightweight yet powerful multilingual model
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # Comprehensive handling of nulls, None, and empty strings
    clean_sentences = (
        df[column_name]
        .astype(str)
        .tolist()
    )
    
    # Generate embeddings (Vectorization)
    embeddings = model.encode(
        clean_sentences, 
        batch_size=batch_size, 
        show_progress_bar=True, 
        convert_to_numpy=True
    )
    
    # Create descriptive column names for the embedding dimensions
    embedding_colnames = [f"emb_sbert_{i}" for i in range(embeddings.shape[1])]
    
    # Construct the embeddings dataframe preserving the original index
    embeddings_df = pd.DataFrame(embeddings, columns=embedding_colnames, index=df.index)
    
    # Concatenate only specific columns: demandpk, literal_reason and all embeddings
    id_cols = [c for c in ['demandpk', column_name] if c in df.columns]
    result_df = pd.concat([df[id_cols], embeddings_df], axis=1)
    
    print(f"Successfully generated {embeddings.shape[1]:,}-dimensional embeddings.")

    # Export to Parquet as we know run_embedding_sbert is enabled
    output_path = os.path.join(settings.source_tables_path, settings.table_embedding_sbert_path)
    result_df.to_parquet(output_path, index=False)
    
    return result_df

In [ ]:
# Call the function
df_embedding_sbert = get_text_embeddings(df_triaje)

In [ ]:
if 'df_embedding_sbert' in locals() and df_embedding_sbert is not None:
    df_embedding_sbert.info(verbose=True, show_counts=True)

In [ ]:
if 'df_embedding_sbert' in locals() and df_embedding_sbert is not None:
    df_embedding_sbert.head()

In [ ]:
# # Merge embedding columns with main table
# df_triaje = dp.merge_text_embeddings(
#     run_mode = settings.run_embedding_sbert,
#     df_embeddings = df_embedding_sbert,
#     embedding_path = settings.table_embedding_sbert_path,
#     df_main = df_triaje
# )

In [ ]:
# df_triaje.info(verbose=True, show_counts=True)

In [ ]:
df_triaje.head()

## Free text - riboBERTa embedding

In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
import pandas as pd


def get_rigoberta_embeddings(
    df,
    column_name='literal_reason',
    batch_size=64,          # Increased: 54 GB RAM (CPU) is well above requirements
    tokenizer=None,         # Accept pre-loaded tokenizer to avoid reloading on repeated calls
    model=None              # Accept pre-loaded model to avoid reloading on repeated calls
):
    """
    Generate specialized clinical embeddings using RigoBERTa-Clinical (IIC).
    Execution is performed locally to ensure the privacy of sensitive emergency data.

    Null handling is delegated to the upstream pipeline (not performed here).

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing 'demandpk' and the text column.
        NaN values in column_name must be imputed before calling this function.
    column_name : str
        The name of the 'literal reason' column to process.
    batch_size : int
        Number of samples per batch.
        64 is recommended for CPU with 54 GB RAM and short clinical texts.
        The dominant cost is compute (matrix multiplications), not memory.
    tokenizer : PreTrainedTokenizer, optional
        Pre-loaded tokenizer. If None, it will be loaded from model_id.
        Pass a pre-loaded instance to avoid reloading when calling this
        function multiple times (e.g. during testing or iterative pipelines).
    model : PreTrainedModel, optional
        Pre-loaded model. If None, it will be loaded from model_id.
        Same rationale as tokenizer parameter above.

    Returns
    -------
    pd.DataFrame or None
        A dataframe with [demandpk, column_name, emb_rigobert_0 ... emb_rigobert_1023].
        Returns None if disabled in settings (run_embedding_rigobert == 0).
    """

    # Check if the process is enabled in the configuration settings
    if settings.run_embedding_rigobert == 0:
        print("--- Skipping RigoBERTa embedding generation as per settings ---")
        return None

    # Select execution device (GPU if available, else CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_id = 'IIC/RigoBERTa-Clinical'

    print(f"--- Generating RigoBERTa embeddings ({device}) for {len(df):,} rows ---")

    # Load tokenizer and model only if not provided externally
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
    if model is None:
        model = AutoModel.from_pretrained(model_id).to(device)

    model.eval()  # Set to evaluation mode for inference (disables dropout, etc.)

    # Null handling delegated to upstream pipeline
    sentences = df[column_name].astype(str).tolist()

    all_embeddings = []  # Collect batch results before consolidation

    # Batch processing with a progress bar (tqdm)
    for i in tqdm(range(0, len(sentences), batch_size), desc="Encoding RigoBERTa"):
        batch = sentences[i : i + batch_size]

        # Tokenize inputs (RigoBERTa-Clinical supports up to 512 tokens)
        # For short emergency texts (typically 2-8 real tokens), padding will
        # dominate each batch — this makes larger batch sizes very RAM-efficient
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

            # MEAN POOLING: Average final hidden states weighted by the attention mask.
            # Tokens corresponding to padding (mask=0) are excluded from the average.
            # This is critical for RigoBERTa to generate high-quality sentence vectors.
            last_hidden = outputs.last_hidden_state          # (batch, seq_len, hidden_size)
            mask_expanded = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()

            sum_embeddings = torch.sum(last_hidden * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

            batch_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
            all_embeddings.append(batch_embeddings)

    # Consolidate all processed batches into a single 2D array: (n_samples, 1024)
    embeddings_array = np.vstack(all_embeddings)

    # FIX: shape[1] (hidden_size dimension), not shape[2] — array is 2D after vstack
    n_dims = embeddings_array.shape[1]
    embedding_colnames = [f"emb_rigobert_{i}" for i in range(n_dims)]

    # Construct result dataframe preserving the original index
    embeddings_df = pd.DataFrame(embeddings_array, columns=embedding_colnames, index=df.index)

    # Select identification columns (demandpk and original text)
    id_cols = [c for c in ['demandpk', column_name] if c in df.columns]
    result_df = pd.concat([df[id_cols], embeddings_df], axis=1)

    print(f"Success: Generated {n_dims:,}-dimensional vectors for {len(result_df):,} rows.")

    # Export the final result to a Parquet file for efficient storage
    output_path = os.path.join(settings.source_tables_path, settings.table_embedding_rigobert_path)
    result_df.to_parquet(output_path, index=False)

    return result_df

In [ ]:
# Call the function
df_embedding_rigobert = get_rigoberta_embeddings(df_triaje)

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.sample(5)

In [ ]:
df_triaje.info(max_cols=500)

# Consort diagram

In [ ]:
# Print excluded rows consecutively
print(f"Rows from original dataset: {n_rows_source_data:,}")
print(f"\nRows excluded by demand type filter (not stroke): {n_rows_excluded_demand_filter:,}")
print(f"Rows from stroke cohort (demand type 54): {n_rows_after_demand_filter:,}")
print(f"Rows excluded with duplicated demandpk: {n_rows_excluded_demandpk:,}")
print(f"Rows excluded in age column: {n_rows_excluded_age:,}")
print(f"Rows excluded with missing values in sex: {n_rows_excluded_sex:,}")
# print(f"Rows excluded with missing values in coordinates: {n_row_excluded_coords:,}")
print(f"Rows excluded with missing values in alert receiver: {n_rows_excluded_alert:,}")
print(f"Rows excluded with trauma diagnoses in ICD: {n_rows_excluded_icd:,}")
print(f"Rows excluded with no specific time: {n_rows_excluded_time:,}")
print(f"Rows excluded without ICD codes in code resolution 31 (found death): {n_rows_excluded_code31:,}")
print(f"Rows excluded without ICD codes at emergency (p1_real_emerg): {n_rows_excluded_p1real:,}")

n_rows_excluded_total = (
    n_rows_excluded_demand_filter +
    n_rows_excluded_demandpk + 
    n_rows_excluded_age + 
    n_rows_excluded_sex + 
    # n_row_excluded_coords +
    n_rows_excluded_alert +
    n_rows_excluded_icd + 
    n_rows_excluded_time +
    n_rows_excluded_code31 +
    n_rows_excluded_p1real
)

# Rows excluded by the eligibility/quality criteria applied within the stroke cohort
# (i.e. all exclusions except the initial demand type filter)
n_rows_excluded_within_stroke = n_rows_excluded_total - n_rows_excluded_demand_filter

print(f"\nRows from original dataset: {n_rows_source_data:,}")
print(f"Total rows excluded: {n_rows_excluded_total:,}")
print(f"% rows excluded (over stroke cohort): {((n_rows_excluded_within_stroke/n_rows_after_demand_filter)*100).round(2):,}")
print(f"Final rows in df_triaje: {len(df_triaje):,}\n")

if len(df_triaje) == (n_rows_source_data - n_rows_excluded_total):
    print("✅ THE LOGIC FOR EXCLUSION CRITERIA IS OK")
else:
    diff = len(df_triaje) - (n_rows_source_data - n_rows_excluded_total)
    print(f"❌ THERE IS AN ERROR CALCULATING EXCLUSION CRITERIA. Difference: {diff:,} rows")

dp.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
from matplotlib.patches import FancyBboxPatch

# --- Layout constants ---
DPI = 300
CX, CW = 0.5, 6.5    # main flow column: left x, width
SX, SW = 8.0, 5.0    # exclusion column: left x, width
BOX_H = 1.0          # default main box height
SOURCE_BOX_H = 1.35 # outcome-source box height
GAP_SHORT = 1.0      # vertical gap for single-criterion transitions
GAP_LONG = 2.4       # vertical gap for the combined (multi-criterion) transition
SIDE_H_SHORT = 0.85  # height of single-criterion exclusion boxes
SIDE_H_LONG = 2.0    # height of the combined exclusion box

FS_MAIN = 13
FS_SIDE = 12
FS_TITLE = 15

W = SX + SW + 0.5
X_CENTER = CX + CW / 2


def box(x, y, w, h, text, fontsize=FS_MAIN, fontweight="normal", facecolor="white", ha="center"):
    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.08",
        linewidth=1.2, edgecolor="black", facecolor=facecolor,
    )
    ax.add_patch(rect)
    tx = x + w / 2 if ha == "center" else x + 0.15
    ax.text(tx, y + h / 2, text, ha=ha, va="center",
            fontsize=fontsize, fontweight=fontweight)


def arrow_v(x, y_from, y_to):
    ax.annotate("", xy=(x, y_to), xytext=(x, y_from),
                 arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))


def arrow_h(x_from, x_to, y):
    ax.annotate("", xy=(x_to, y), xytext=(x_from, y),
                 arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))


# --- Cohort sizes after each step ---
n0 = n_rows_source_data
n1 = n0 - n_rows_excluded_demand_filter
n8 = (
    n1
    - n_rows_excluded_demandpk
    - n_rows_excluded_age
    - n_rows_excluded_sex
    - n_rows_excluded_alert
    - n_rows_excluded_icd
    - n_rows_excluded_time
    - n_rows_excluded_code31
)
n9 = n8 - n_rows_excluded_p1real

# Outcome sources are not mutually exclusive; counts are computed in the
# final analytical cohort, where p1_real_emerg_bps is available.
n_emerg_source = int(df_triaje["p1_real_emerg"].notna().sum())
n_bps_source = int(df_triaje["p1_real_bps"].notna().sum())
n_both_sources = int(
    (df_triaje["p1_real_emerg"].notna() & df_triaje["p1_real_bps"].notna()).sum()
)

stages = [
    f"Unique patient-calls in EMCC in 2020-2024\n(n = {n0:,})",
    f"Suspected acute stroke cohort (code A54)\n(n = {n1:,})",
    f"Cohort after inclusion/exclusion criteria\n(n = {n8:,})",
    "Outcome source:\n"
    f"• Emergency diagnosis available: n = {n_emerg_source:,}\n"
    f"• Hospital diagnosis available: n = {n_bps_source:,}\n"
    f"• Both sources available: n = {n_both_sources:,}",
    f"Final cohort for analysis\n(n = {n9:,})",
]
stage_heights = [BOX_H, BOX_H, BOX_H, SOURCE_BOX_H, BOX_H]

# Combined count for the second exclusion box, itemized excluding zero-count criteria
n_rows_excluded_criteria = (
    n_rows_excluded_demandpk
    + n_rows_excluded_age
    + n_rows_excluded_sex
    + n_rows_excluded_alert
    + n_rows_excluded_icd
    + n_rows_excluded_time
    + n_rows_excluded_code31
)

criteria_items = [
    ("Duplicate demandpk (≥2 patients)", n_rows_excluded_demandpk),
    ("Age greater than 105 years", n_rows_excluded_age),
    ("Missing sex", n_rows_excluded_sex),
    ("Missing alert receiver", n_rows_excluded_alert),
    ("Trauma diagnoses", n_rows_excluded_icd),
    ("No valid contact/activation time", n_rows_excluded_time),
    ("Resolution code 31 (found death) without ICD", n_rows_excluded_code31),
]
criteria_items = [(label, n) for label, n in criteria_items if n > 0]

exclusions = [
    f"Excluded: other demand types\n(not stroke)\nn = {n_rows_excluded_demand_filter:,}",
    f"Excluded (n = {n_rows_excluded_criteria:,}):\n" + "\n".join(
        f"• {label}: {n:,}" for label, n in criteria_items
    ),
    f"Excluded:\nno registered final ICD\nn = {n_rows_excluded_p1real:,}",
    None,
]

side_heights = [SIDE_H_SHORT, SIDE_H_LONG, SIDE_H_SHORT, None]
side_fontsizes = [FS_SIDE, FS_SIDE - 1, FS_SIDE, None]
side_ha = ["center", "left", "center", None]
gaps = [GAP_SHORT, GAP_LONG, GAP_SHORT, 0.7]

H = sum(stage_heights) + sum(gaps) + 1.0

fig, ax = plt.subplots(figsize=(12, 11))
ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.axis("off")
ax.invert_yaxis()

# --- Draw main flow, top to bottom, with exclusion boxes branching from the central axis ---
y_top = 0.5
for i, label in enumerate(stages):
    stage_h = stage_heights[i]
    is_endpoint = i in (0, len(stages) - 1)
    is_source_box = i == len(stages) - 2
    box(
        CX, y_top, CW, stage_h, label,
        fontsize=FS_MAIN - 1 if is_source_box else FS_MAIN,
        fontweight="bold" if is_endpoint else "normal",
        facecolor="#d9e6f2" if is_endpoint else "white",
    )

    if i < len(stages) - 1:
        gap = gaps[i]
        next_y_top = y_top + stage_h + gap
        arrow_v(X_CENTER, y_top + stage_h, next_y_top)

        if exclusions[i] is not None:
            y_mid = y_top + stage_h + gap / 2
            side_h = side_heights[i]
            box(SX, y_mid - side_h / 2, SW, side_h, exclusions[i],
                fontsize=side_fontsizes[i], facecolor="#fbe5e5", ha=side_ha[i])
            arrow_h(X_CENTER, SX, y_mid)

        y_top = next_y_top

ax.set_title(
    "CONSORT-style flow diagram of inclusion and exclusion criteria",
    fontsize=FS_TITLE, fontweight="bold", pad=20,
)

fig_dir = Path(settings.triaje_project_path) / "figures" / "04_stroke" / "consort_diagram"
fig_dir.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(fig_dir / "consort_flowchart.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
print(f"\nSample size for unique patient-calls with emergency diagnosis: {df_triaje.shape[0]}")

df_triaje["p1_real_emerg"].value_counts(dropna=False)

In [ ]:
df_triaje["p1_real_emerg_bps"].value_counts(dropna=False)

In [ ]:
df_triaje["p1_real_bps"].value_counts(dropna=False)

# Data Export

In [ ]:
# Save the cleaned dataframe to a Parquet file
df_triaje.to_parquet(os.path.join(settings.source_tables_path, settings.triaje_table_cleaned_path), index=False)

# Get final dimensions of final preprocessed dataset
final_rows = df_triaje.shape[0]
final_cols = df_triaje.shape[1]

# Define the path and check if the file was created correctly
export_path = os.path.join(settings.source_tables_path, settings.triaje_table_cleaned_path)
if os.path.exists(export_path):
    # Calculate file size in MB
    file_size_mb = os.path.getsize(export_path) / (1024 * 1024)
       
    print(f"✅ Export successful!")
    print(f"--- Path: {export_path}")
    print(f"--- Size: {file_size_mb:,.2f} MB")
    print(f"--- Final Rows: {final_rows:,}")
    print(f"--- Final Columns: {final_cols:,}")
else:
    print(f"❌ Export failed: File not found at {export_path}")

ml.time_print()

In [ ]:
import psutil
import torch
import os

def check_resources():
    """
    Checks and prints the available CPU RAM and GPU VRAM resources.
    Useful for determining the optimal batch_size for embedding generation.
    """
    print("=== System Resource Report ===")
    
    # 1. CPU RAM Monitoring
    vm = psutil.virtual_memory()
    total_ram = vm.total / (1024**3)
    available_ram = vm.available / (1024**3)
    used_ram_pct = vm.percent
    
    print(f"\n[CPU RAM]")
    print(f"  Total:     {total_ram:,.2f} GB")
    print(f"  Available: {available_ram:,.2f} GB")
    print(f"  Usage:     {used_ram_pct}%")

    # 2. GPU VRAM Monitoring (NVIDIA)
    print(f"\n[GPU / CUDA]")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            total_vram = props.total_memory / (1024**3)
            # Memory currently allocated by PyTorch
            allocated_vram = torch.cuda.memory_allocated(i) / (1024**3)
            # Memory reserved by PyTorch but not necessarily used
            reserved_vram = torch.cuda.memory_reserved(i) / (1024**3)
            # Rough estimate of free VRAM
            free_vram = total_vram - reserved_vram
            
            print(f"  Device {i}: {props.name}")
            print(f"    Total VRAM:     {total_vram:,.2f} GB")
            print(f"    Allocated VRAM: {allocated_vram:,.2f} GB")
            print(f"    Free VRAM (est): {free_vram:,.2f} GB")
    else:
        print("  CUDA is not available. Using CPU only.")

    # 3. Process Specific Usage
    process = psutil.Process(os.getpid())
    process_ram = process.memory_info().rss / (1024**3)
    print(f"\n[Current Jupyter Process]")
    print(f"  RAM Usage: {process_ram:,.2f} GB")

if __name__ == "__main__":
    check_resources()

In [ ]:
import gc
import pandas as pd

def clear_environment():
    """
    Deletes all DataFrames and user-defined variables from the global scope
    and forces garbage collection to free up physical RAM.
    """
    # 1. Get a list of all global variable names
    all_vars = list(globals().keys())
    
    # 2. Iterate and delete
    for var_name in all_vars:
        # We exclude protected/internal variables and essential modules
        if not var_name.startswith('_') and var_name not in ['pd', 'np', 'gc', 'plt', 'clear_environment']:
            if var_name in globals():
                del globals()[var_name]
    
    # 3. Force the Garbage Collector to release the memory
    gc.collect()
    
    print("--- Environment cleared & RAM released ---")

# Execute the cleanup
clear_environment()

# EDA

## Tests

In [ ]:
df_triaje.info(verbose=True, show_counts=True)

In [ ]:
df_triaje.head(5)

In [ ]:
df_triaje['p1_real_emerg'].value_counts(dropna = False)

In [ ]:
df_triaje['comment'].value_counts(dropna = False)

In [ ]:
df_triaje['demand_type_2'].value_counts(dropna = False)


In [ ]:
df_triaje['demand_type_3'].value_counts(dropna = False)

In [ ]:
target_analysis_report = analyze_p1_missing_by_demand_type(df_triaje)
print("=== Target (p1_real_emerg) Missing Analysis by Demand Type ===")
print(target_analysis_report)

In [ ]:
df_triaje['cadrec'].value_counts(dropna = False)

In [ ]:
icd_cols = ["icd1", "icd2", "icd3"]

df_filtered = dp.filter_rows_by_icd(
    df=df_triaje,
    icd_value="047.9",
    icd_columns=icd_cols
)

df_filtered.head()

In [ ]:
# Filter rows where 'icd_emerg_combined' contains 'z51.5' using lowercase comparison
df_triaje_icd = df_triaje[df_triaje["icd_emerg_combined"].str.lower().str.contains("z51.5", na=False)]

# Print the number of rows that match the filter
print(f"Number of rows found for ICD code 'z51.5': {len(df_triaje_icd):,}")

df_triaje_icd['p1_real_emerg'].value_counts(dropna = False)

In [ ]:
# Normalize ICD columns to lowercase strings
df_icd = (
    df_triaje[["icd1", "icd2", "icd3"]]
    .astype("string")
    .apply(lambda col: col.str.lower())
)

# Build mask: row contains ICD 'z51.5' in any of the three ICD columns
mask_z515 = df_icd.eq("z51.5").any(axis=1)

# Filter rows
df_triaje_icd_z515 = df_triaje[mask_z515]

# Print number of matches
print(f"Number of rows found for ICD 'z51.5': {len(df_triaje_icd_z515):,}")

# Inspect distribution of P1
df_triaje_icd_z515["p1_real_emerg"].value_counts(dropna=False)


In [ ]:
# --- Método substring ---
mask_substring = df_triaje["icd_emerg_combined"].str.lower().str.contains("z51.5", na=False)

# --- Método exacto ---
df_icd = (
    df_triaje[["icd1", "icd2", "icd3"]]
    .astype("string")
    .apply(lambda col: col.str.lower())
)

mask_exact = df_icd.eq("z51.5").any(axis=1)

# --- Filas detectadas SOLO por método substring ---
mask_collision = mask_substring & (~mask_exact)

df_collision = df_triaje[mask_collision]

print(f"Rows detected by substring but NOT exact match: {len(df_collision):,}")
df_collision[["demandpk", "icd_emerg_combined", "icd1", "icd2", "icd3"]].head(20)


In [ ]:
# Filter rows where 'icd_emerg_combined' contains 'v66.7' using lowercase comparison
df_triaje_icd = df_triaje[df_triaje["icd_emerg_combined"].str.lower().str.contains("v66.7", na=False)]

# Print the number of rows that match the filter
print(f"Number of rows found for ICD code 'v66.7': {len(df_triaje_icd):,}")

df_triaje_icd['p1_real_emerg'].value_counts(dropna = False)

In [ ]:
# Normalize ICD columns to lowercase strings
df_icd = (
    df_triaje[["icd1", "icd2", "icd3"]]
    .astype("string")
    .apply(lambda col: col.str.lower())
)

# Build mask: row contains ICD 'v66.7'
mask_v667 = df_icd.eq("v66.7").any(axis=1)

# Filter rows
df_triaje_icd_v667 = df_triaje[mask_v667]

# Print number of matches
print(f"Number of rows found for ICD 'v66.7': {len(df_triaje_icd_v667):,}")

# Inspect distribution of P1
df_triaje_icd_v667["p1_real_emerg"].value_counts(dropna=False)


In [ ]:
mask_both = (
    df_triaje["icd_emerg_combined"].str.lower().str.contains("z51.5", na=False)
    &
    df_triaje["icd_emerg_combined"].str.lower().str.contains("v66.7", na=False)
)

df_both = df_triaje[mask_both]

print(f"Rows containing BOTH ICDs (z51.5 AND v66.7): {len(df_both):,}")
print(df_both["p1_real_emerg"].value_counts(dropna=False))


In [ ]:
patterns = ["z51.5", "v66.7"]
df_icd_clean = df_triaje[["icd1", "icd2", "icd3"]].astype("string").apply(lambda col: col.str.lower())

mask_exact = (
    df_icd_clean.eq("z51.5").any(axis=1)
    &
    df_icd_clean.eq("v66.7").any(axis=1)
)

df_both = df_triaje[mask_exact]

print(f"Rows containing BOTH ICDs exactly (z51.5 AND v66.7): {len(df_both):,}")
print(df_both["p1_real_emerg"].value_counts(dropna=False))


In [ ]:
# Simple sample of 500 rows for demand type 54
pd.set_option('display.max_rows', None)
sample_list = df_triaje[df_triaje["demand_type_1"] == 54]["literal_reason"].sample(2000).tolist()

In [ ]:
# 2. Print each element line by line
for i, text in enumerate(sample_list, 1):
    print(f"{text}")

## Resolution codes

In [ ]:
import re

# -------------------------------------------------------
# 1. Split combined codes (e.g. "34 | 28") into one code per row
# -------------------------------------------------------

# Split by '|' and clean whitespace
df_codes = (
    df_triaje['resolution_code_id']
    .astype(str)
    .str.split('|')
    .apply(lambda x: [item.strip() for item in x])  # remove spaces
    .explode()                                      # create one row per code
    .replace({'': None, 'nan': None, 'None': None}) # normalize missing values
)

# -------------------------------------------------------
# 2. Helper function to sort by numeric + alphabetical part
# -------------------------------------------------------
def split_code(code):
    """
    Splits a resolution code into numeric and alphabetical parts.
    Example: "34A" -> (34, "A"), "36" -> (36, "")
    """
    if code is None:
        return (float('inf'), "")  # send None to the bottom
    
    code = str(code).strip()
    
    # Capture digits + optional letters
    match = re.match(r"(\d+)([A-Za-z]*)", code)
    if match:
        num = int(match.group(1))
        let = match.group(2)
        return (num, let)
    else:
        # If unexpected format, send it to the bottom
        return (float('inf'), code)

# -------------------------------------------------------
# 3. Unique values sorted
# -------------------------------------------------------
unique_codes = df_codes.dropna().unique()
sorted_codes = sorted(unique_codes, key=split_code)

# -------------------------------------------------------
# 4. Value counts sorted according to numeric+letter order
# -------------------------------------------------------
vc = df_codes.value_counts(dropna=False)

# Create sorted index
sorted_index = sorted(vc.index, key=split_code)

# Reindex to apply sorted order
vc_sorted = vc.reindex(sorted_index)

# -------------------------------------------------------
# 5. Display results
# -------------------------------------------------------
vc_sorted


# DATA QUALITY

## Analysis of Missing

In [ ]:
dp.analyze_missing_values(df_triaje)